# Les GAB du parc : un modèle d'intelligence artificielle pour repérer chaque jour les GAB atypiques

**À quoi sert ce notebook.** Il reprend le premier prototype d'intelligence artificielle (qui travaillait au mois) et le fait tourner
**jour par jour** sur la fiche d'identité des GAB. Chaque jour, il produit une liste courte des GAB les plus atypiques, avec une raison
claire pour chacun.

**Ce qui a changé par rapport au premier prototype, et pourquoi**

- **Au jour plutôt qu'au mois** : les incidents recherchés (inversion de cassettes, GAB à l'arrêt, rafale de cartes étrangères) durent un
  jour ou quelques heures. Sur un mois, ils disparaissent dans la moyenne.
- **Chaque GAB comparé à ses habitudes** : un petit GAB qui s'arrête ou un GAB qui triple ses retraits se voit, alors qu'en valeur brute il
  ressemble à d'autres GAB.
- **Chaque GAB jugé parmi ses pareils** : les GAB sont rangés en familles, et un GAB n'est comparé qu'aux GAB de sa famille.
- **Une liste courte** : 10 à 15 GAB par jour, jamais la moitié du parc.
- **Une raison en clair** pour chaque GAB signalé.

**Comment le lire.** Chaque section dit ce que vous voyez et à quoi ça sert. Les détails techniques sont dans des encadrés « Comment c'est
calculé », à ouvrir si besoin. Le fichier Excel (section 11) reprend tous les résultats.

| Mot | Ce qu'il veut dire |
|---|---|
| **GAB-jour** | un GAB, un jour donné. Un GAB atypique pendant 10 jours compte 10 GAB-jours, mais un seul GAB |
| **Habitude** | ce que fait d'ordinaire **ce GAB** ce jour-là de la semaine |
| **Rareté** | « 1 sur 1 000 » : une journée aussi inhabituelle n'arrive qu'une fois sur 1 000 |
| **Rang du jour** | le classement du GAB parmi tout le parc ce jour-là : 1 = le plus atypique |

| Partie | Section | Ce qu'on y trouve |
|---|---|---|
| | 1 | Réglages (pour la personne qui lance le notebook) |
| A. Les données | 2 | Jusqu'à quelle date les données sont-elles fiables ? |
| | 3 | Ce que le modèle regarde pour chaque GAB, chaque jour |
| | 4 | Journées ordinaires et journées suspectes ; cartes étrangères |
| | 5 | Les rythmes du parc |
| B. Les familles | 6 | Les types de GAB ; les GAB qui changent de comportement |
| C. Le modèle | 7 | Comment le modèle classe les GAB chaque jour |
| | 8 | Pourquoi chaque GAB est signalé ; les chiffres du jour |
| | 9 | Le dernier jour : les GAB à regarder ; les GAB qui reviennent souvent |
| | 10 | Le modèle retrouve-t-il les incidents connus ? |
| D. Les livrables | 11 | Le bilan par année et le fichier Excel |

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s0"></a>
## 0. Préparation technique (rien à lire)
Rien à modifier ici. Cette cellule charge les bibliothèques et définit les quelques fonctions qui mettent en forme
les résultats (encadrés, tuiles de chiffres, tableaux, graphiques). Les graphiques sont interactifs (Plotly : survoler
un point affiche sa valeur) ; si Plotly manque, les mêmes graphiques sont dessinés avec matplotlib.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import base64
import unicodedata
import zipfile
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import dataiku
from IPython.display import display, HTML

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)

# --- Plotly (interactif) si disponible, sinon matplotlib
try:
    import plotly.graph_objs as go
    from plotly.offline import init_notebook_mode, iplot
    init_notebook_mode(connected=False)          # la bibliothèque est embarquée dans le notebook : fonctionne sans internet
    PLOTLY = True
except Exception as _e:
    PLOTLY = False
    print("Plotly indisponible ({}) : graphiques matplotlib.".format(_e))

# --- Palette unique pour tout le notebook
BLEU, BLEU_CLAIR, GRIS, GRIS_CLAIR, NOIR = "#2f6db5", "#9dbfe3", "#8a8f98", "#d9d8d3", "#1f2328"
COULEURS_NIVEAU = {"Critique": "#cf222e", "Élevé": "#e36209", "À surveiller": "#c99a06", "Normal": "#b8b8b8"}
ORDRE_NIVEAUX = ["Critique", "Élevé", "À surveiller", "Normal"]
SERIE = [BLEU, "#e36209", "#2da44e", "#8250df", "#bf3989", "#1b7c83", "#9a6700", "#57606a"]

CSS = """<style>
.g-note{border-left:4px solid #2f6db5;background:#f6f8fa;padding:8px 12px;margin:8px 0;font-size:13.5px;line-height:1.45;color:#1f2328}
.g-note.warn{border-color:#d29922;background:#fff8e6}.g-note.bad{border-color:#cf222e;background:#fff1f0}.g-note.ok{border-color:#1a7f37;background:#eefbf1}
.g-ret{border:1px solid #d0d7de;border-left:6px solid #2f6db5;border-radius:4px;background:#fff;padding:10px 14px;margin:10px 0;font-size:13.5px;line-height:1.5}
.g-ret b.t{display:block;color:#2f6db5;text-transform:uppercase;font-size:11px;letter-spacing:.06em;margin-bottom:4px}
.g-ret ul{margin:4px 0 0 18px;padding:0}.g-ret li{margin:2px 0}
.g-band{background:#1f2328;color:#fff;padding:10px 16px;margin:14px 0 6px 0;border-radius:4px}
.g-band b{font-size:17px;letter-spacing:.02em}.g-band span.sub{display:block;color:#c9d1d9;font-size:12.5px;margin-top:4px}
.g-kpis{display:flex;flex-wrap:wrap;gap:10px;margin:8px 0}
.g-kpi{flex:1 1 150px;min-width:150px;max-width:230px;border:1px solid #d0d7de;border-radius:6px;padding:10px 12px;background:#fff}
.g-kpi .l{font-size:11.5px;color:#57606a;text-transform:uppercase;letter-spacing:.04em}
.g-kpi .v{font-size:24px;font-weight:600;color:#1f2328;margin:2px 0;font-variant-numeric:tabular-nums}
.g-kpi .s{font-size:12px;color:#57606a}.g-kpi .s.up{color:#1a7f37}.g-kpi .s.down{color:#cf222e}
table.g{border-collapse:collapse;border:0 !important;font-size:12.5px;margin:6px 0;font-family:inherit}
table.g caption{text-align:left;font-weight:600;font-size:13.5px;padding:4px 0 6px 0;color:#1f2328}
table.g th{background:#f6f8fa;color:#1f2328;text-align:left !important;padding:6px 10px;border-bottom:2px solid #d0d7de;font-weight:600;white-space:nowrap}
table.g tbody tr td,table.g tr td{padding:5px 10px;border-bottom:1px solid #eaeef2;vertical-align:top;text-align:left !important}table.g tr:nth-child(even) td{background:#fbfcfd}
table.g tbody tr td.n,table.g tr td.n{text-align:right !important;font-variant-numeric:tabular-nums;white-space:nowrap}table.g tr td.d{max-width:620px;line-height:1.4}
table.g tr td.v{max-width:200px;color:#57606a;font-size:11.5px}table.g tr td.m{color:#57606a;font-size:11.5px}
.pill{display:inline-block;padding:1px 8px;border-radius:10px;font-size:11.5px;font-weight:600;color:#fff;white-space:nowrap}
.pill.c{background:#cf222e}.pill.e{background:#e36209}.pill.s{background:#c99a06}.pill.n{background:#b8b8b8;color:#1f2328}.pill.i{background:#8a8f98}
.g-leg{font-size:12px;color:#57606a;margin:2px 0 8px 0}
</style>"""
display(HTML(CSS))
# bouton « Afficher / masquer le code » : les lecteurs du métier voient les textes, les tableaux et les graphiques sans le code
display(HTML("""<style>div.code_cell div.input, .jp-CodeCell .jp-Cell-inputWrapper {display:none}</style>
<button onclick="var v = this.getAttribute('data-v') != '1'; this.setAttribute('data-v', v ? '1' : '0');
  document.querySelectorAll('div.code_cell div.input, .jp-CodeCell .jp-Cell-inputWrapper').forEach(function (z) { z.style.display = v ? 'flex' : 'none'; });
  this.innerText = v ? 'Masquer le code' : 'Afficher le code';"
  style="padding:6px 14px;border:1px solid #2f6db5;border-radius:6px;background:#fff;color:#2f6db5;font-weight:600;cursor:pointer">Afficher le code</button>
<span style="font-size:12px;color:#57606a;margin-left:8px">Le code est masqué pour la lecture : ce bouton le fait réapparaître.</span>"""))


CLASSE_NIVEAU = {"Critique": "c", "Élevé": "e", "À surveiller": "s", "Normal": "n", "information": "i"}


def jfr(d, court=False):
    """Date en français : « lundi 08/11/2026 » (ou « lun. 08/11 »)."""
    d = pd.Timestamp(d)
    j = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"][d.dayofweek]
    return (j[:3] + ". " + d.strftime("%d/%m")) if court else (j + " " + d.strftime("%d/%m/%Y"))


def nb(x, dec=0):
    """Nombre lisible : 12 345 ou 12 345,6 ; '-' si manquant."""
    if x is None or (isinstance(x, float) and np.isnan(x)) or (not isinstance(x, (int, float, np.integer, np.floating))):
        return "-" if x is None or (isinstance(x, float) and np.isnan(x)) else str(x)
    s = "{:,.{}f}".format(float(x), dec).replace(",", " ").replace(".", ",")
    return s


def pct(x, dec=1):
    return "-" if x is None or pd.isnull(x) else ("{:.{}f} %".format(100.0 * x, dec)).replace(".", ",")


EN_CLAIR = [(" (loi de Poisson)", ""), ("écart ≥ 3 variations habituelles", "nettement au-dessus de ses variations habituelles"),
            ("écart ≤ −3 variations habituelles", "nettement en dessous de ses variations habituelles"), (" (écart médian ≥ 2)", ""), (" (écart médian ≤ −2)", ""),
            ("distance de Jensen-Shannon ≥ 0,25 entre la répartition horaire du jour et celle des 28 derniers jours", "les retraits se répartissent dans la journée très différemment des 4 dernières semaines"),
            ("distance de Jensen-Shannon ≥ 0,25 entre la répartition par réseau du jour et celle des 28 derniers jours", "les types de cartes utilisés sont très différents des 4 dernières semaines"),
            ("vu la moyenne des 28 derniers jours", "vu les 4 dernières semaines du GAB"), ("(écart ≥ 3)", "(très au-dessus de l'habitude)"), (" (écart ≥ 4)", ""),
            ("et écart ≥ 3", ", très au-dessus de l'habitude"), ("médiane des GAB-jours actifs", "valeur typique d'un jour de GAB"), ("valeurs médianes", "valeurs typiques"), ("(médiane des 4 précédents)", "(valeur typique des 4 précédents)"), ("(médiane)", "(valeur typique)"),
            ("médiane", "valeur typique"), ("GAB médian", "GAB typique"), ("volume médian", "volume typique"), ("distance ", "écart de répartition ")]


def en_clair(t):
    """Remplace le vocabulaire technique par des mots de tous les jours."""
    t = str(t)
    for a_, b_ in EN_CLAIR:
        t = t.replace(a_, b_)
    return t


def note(texte, genre="info"):
    """Encadré court : info (bleu), warn (orange), bad (rouge), ok (vert)."""
    display(HTML(CSS + '<div class="g-note {}">{}</div>'.format("" if genre == "info" else genre, texte)))


def bandeau(titre, sous_titre=""):
    display(HTML(CSS + '<div class="g-band"><b>{}</b><span class="sub">{}</span></div>'.format(en_clair(titre), en_clair(sous_titre))))


def retenir(points, titre="À retenir"):
    """Encadré « À retenir » : liste de phrases, calculées sur les données."""
    display(HTML(CSS + '<div class="g-ret"><b class="t">{}</b><ul>{}</ul></div>'.format(titre, "".join("<li>{}</li>".format(en_clair(p)) for p in points if p))))


def tuiles(items):
    """Tuiles de chiffres : items = [(libellé, valeur, sous-texte, sens)] ; sens = 'up' / 'down' / '' colore le sous-texte."""
    h = ""
    for it in items:
        lib, val = it[0], it[1]
        sous = it[2] if len(it) > 2 else ""
        sens = it[3] if len(it) > 3 else ""
        h += '<div class="g-kpi"><div class="l">{}</div><div class="v">{}</div><div class="s {}">{}</div></div>'.format(en_clair(lib), val, sens, en_clair(sous))
    display(HTML(CSS + '<div class="g-kpis">{}</div>'.format(h)))


def _html(x):
    return str(x).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def tableau(df, titre="", n=50, libelles=None, pills=("niveau",), longues=("diagnostic", "resume", "résumé", "comment on décide", "exemple"),
            petites=("voisins", "anomalies", "code technique", "codes"), index=False, dec=None):
    """Tableau HTML propre : en-têtes en français (dict `libelles`), nombres alignés à droite, niveau en pastille,
    textes longs sur une colonne large, colonnes secondaires en petit. Jamais plus de `n` lignes."""
    t = df.head(n).copy()
    if index:
        t = t.reset_index()
    libelles = libelles or {}
    cols = list(t.columns)
    th = "".join("<th>{}</th>".format(_html(en_clair(libelles.get(c, c)))) for c in cols)
    lignes = []
    for _, r in t.iterrows():
        cel = []
        for c in cols:
            x = r[c]
            lc = str(libelles.get(c, c)).lower()
            if c in pills or lc in pills:
                cel.append('<td><span class="pill {}">{}</span></td>'.format(CLASSE_NIVEAU.get(str(x), "i"), _html(x)))
            elif c in longues or lc in longues:
                cel.append('<td class="d">{}</td>'.format(_html("" if pd.isnull(x) else x)))
            elif c in petites or lc in petites:
                cel.append('<td class="v">{}</td>'.format(_html("" if pd.isnull(x) else x)))
            elif isinstance(x, (pd.Timestamp,)):
                cel.append("<td>{}</td>".format(jfr(x)))
            elif isinstance(x, (float, np.floating)):
                d_ = dec if dec is not None else (0 if abs(x) >= 100 or float(x).is_integer() else 1)
                cel.append('<td class="n">{}</td>'.format(nb(x, d_)))
            elif isinstance(x, (int, np.integer)) and not isinstance(x, (bool, np.bool_)):
                cel.append('<td class="n">{}</td>'.format(nb(x)))
            else:
                sx = "" if pd.isnull(x) else str(x)
                cel.append('<td{}>{}</td>'.format(' style="white-space:nowrap"' if len(sx) <= 28 else "", _html(sx)))
        lignes.append("<tr>{}</tr>".format("".join(cel)))
    cap = "<caption>{}</caption>".format(en_clair(titre)) if titre else ""
    suite = '<div class="g-leg">… {} lignes de plus (voir l\'Excel).</div>'.format(nb(len(df) - n)) if len(df) > n else ""
    return HTML(CSS + '<table class="g">{}<tr>{}</tr>{}</table>{}'.format(cap, th, "".join(lignes), suite))


def pastilles(niveaux):
    """« 3 Critique, 12 Élevé, 40 À surveiller » avec les couleurs."""
    vc = pd.Series(niveaux).value_counts()
    return " ".join('<span class="pill {}">{} {}</span>'.format(CLASSE_NIVEAU[k], nb(vc.get(k, 0)), k) for k in ORDRE_NIVEAUX if vc.get(k, 0))


# --- Graphiques : une fonction par forme ; Plotly si présent, sinon matplotlib
def _layout(titre, x_titre="", y_titre="", hauteur=360, **kw):
    lay = dict(title=dict(text=en_clair(titre), x=0.01, font=dict(size=15, color=NOIR)), height=hauteur, margin=dict(l=50, r=20, t=50, b=45),
               paper_bgcolor="#fff", plot_bgcolor="#fff", font=dict(family="Segoe UI, Helvetica, Arial, sans-serif", size=12, color=NOIR),
               xaxis=dict(title=x_titre, gridcolor="#eaeef2", zeroline=False, showline=True, linecolor=GRIS_CLAIR),
               yaxis=dict(title=y_titre, gridcolor="#eaeef2", zeroline=False, showline=False),
               legend=dict(orientation="h", y=1.02, yanchor="bottom", x=1, xanchor="right", font=dict(size=11)), hovermode="closest")
    lay.update(kw)
    return go.Layout(**lay)


def _est_date(x):
    try:
        return pd.api.types.is_datetime64_any_dtype(pd.Series(list(x)[:5]))
    except Exception:
        return False


def montrer(fig):
    iplot(fig, show_link=False)


def g_courbes(x, series, titre, y_titre="", x_titre="", hauteur=360, hover=None, marqueurs=None, bandes=None, lignes_v=None, y_zero=False):
    """Courbes (series = [(nom, valeurs, couleur, style)]) ; marqueurs = [(nom, x, y, couleur)] points ; bandes = [(x0, x1, couleur, nom)] ;
    lignes_v = [(x, nom)]."""
    if PLOTLY:
        data = []
        for i, s_ in enumerate(series):
            nom, y = s_[0], s_[1]
            coul = s_[2] if len(s_) > 2 and s_[2] else SERIE[i % len(SERIE)]
            style = s_[3] if len(s_) > 3 else "solid"
            data.append(go.Scatter(x=list(x), y=list(y), mode="lines", name=nom, line=dict(color=coul, width=1.8 if style == "solid" else 1.4, dash=style),
                                   hovertemplate="%{x|%d/%m/%Y}<br>" + nom + " : %{y:,.0f}<extra></extra>" if hover is None else hover))
        for m in (marqueurs or []):
            data.append(go.Scatter(x=list(m[1]), y=list(m[2]), mode="markers", name=m[0], marker=dict(color=m[3], size=9, line=dict(color="#fff", width=1)),
                                   text=list(m[4]) if len(m) > 4 else None, hovertemplate="%{x|%d/%m/%Y}<br>%{text}<extra>" + m[0] + "</extra>" if len(m) > 4 else None))
        shapes = [dict(type="rect", xref="x", yref="paper", x0=b[0], x1=b[1], y0=0, y1=1, fillcolor=b[2], opacity=0.12, line=dict(width=0)) for b in (bandes or [])]
        shapes += [dict(type="line", xref="x", yref="paper", x0=l[0], x1=l[0], y0=0, y1=1, line=dict(color=NOIR, width=1, dash="dot")) for l in (lignes_v or [])]
        annos = [dict(x=l[0], y=1, yref="paper", text=l[1], showarrow=False, yanchor="bottom", font=dict(size=11)) for l in (lignes_v or [])]
        lay = _layout(titre, x_titre, y_titre, hauteur, shapes=shapes, annotations=annos)
        if _est_date(x):
            lay.xaxis.update(tickformat="%d/%m/%Y", hoverformat="%d/%m/%Y")
        if y_zero:
            lay.yaxis.update(rangemode="tozero")
        montrer(go.Figure(data=data, layout=lay))
    else:
        fig, ax = plt.subplots(figsize=(13, hauteur / 80.0))
        for i, s_ in enumerate(series):
            ax.plot(x, s_[1], label=s_[0], color=s_[2] if len(s_) > 2 and s_[2] else SERIE[i % len(SERIE)], linewidth=1.4,
                    linestyle="--" if len(s_) > 3 and s_[3] != "solid" else "-")
        for m in (marqueurs or []):
            ax.scatter(m[1], m[2], color=m[3], s=30, zorder=3, label=m[0])
        for b in (bandes or []):
            ax.axvspan(b[0], b[1], color=b[2], alpha=0.12)
        for l in (lignes_v or []):
            ax.axvline(l[0], color=NOIR, linewidth=1, linestyle=":")
        ax.set_title(titre, loc="left"); ax.set_ylabel(y_titre); ax.set_xlabel(x_titre); ax.grid(alpha=0.3); ax.legend(fontsize=8, frameon=False)
        plt.tight_layout(); plt.show()


def g_barres(cats, series, titre, y_titre="", x_titre="", hauteur=340, empile=False, horizontal=False, hover_suffixe="", ligne_y=None):
    """Barres : series = [(nom, valeurs, couleur)] ; empile=True pour empiler ; ligne_y = repère horizontal."""
    cats = [str(c) for c in cats]
    if PLOTLY:
        data = []
        for i, s_ in enumerate(series):
            coul = s_[2] if len(s_) > 2 and s_[2] else SERIE[i % len(SERIE)]
            vals = list(s_[1])
            if horizontal:
                data.append(go.Bar(y=cats, x=vals, orientation="h", name=s_[0], marker=dict(color=coul), hovertemplate="%{y}<br>%{x:,.1f}" + hover_suffixe + "<extra>" + s_[0] + "</extra>"))
            else:
                data.append(go.Bar(x=cats, y=vals, name=s_[0], marker=dict(color=coul), hovertemplate="%{x}<br>%{y:,.1f}" + hover_suffixe + "<extra>" + s_[0] + "</extra>"))
        shapes = [dict(type="line", xref="paper", yref="y", x0=0, x1=1, y0=ligne_y, y1=ligne_y, line=dict(color=GRIS, width=1, dash="dash"))] if ligne_y is not None else []
        lay = _layout(titre, x_titre, y_titre, hauteur, barmode="stack" if empile else "group", bargap=0.25, shapes=shapes, showlegend=len(series) > 1)
        if len(series) > 3:          # beaucoup de séries : la légende passe sous le graphique, pour ne pas couvrir le titre
            lay.legend.update(y=-0.12, yanchor="top", x=0, xanchor="left")
            lay.margin.update(b=110)
            lay.height = hauteur + 60
        montrer(go.Figure(data=data, layout=lay))
    else:
        fig, ax = plt.subplots(figsize=(12, hauteur / 80.0))
        bas = np.zeros(len(cats))
        larg = 0.8 / (1 if empile else len(series))
        for i, s_ in enumerate(series):
            coul = s_[2] if len(s_) > 2 and s_[2] else SERIE[i % len(SERIE)]
            vals = np.asarray(list(s_[1]), dtype=float)
            if horizontal:
                ax.barh(cats, vals, left=bas if empile else None, color=coul, label=s_[0])
            else:
                pos = np.arange(len(cats)) + (0 if empile else (i - (len(series) - 1) / 2.0) * larg)
                ax.bar(pos, vals, width=larg, bottom=bas if empile else None, color=coul, label=s_[0])
                ax.set_xticks(np.arange(len(cats))); ax.set_xticklabels(cats, rotation=45 if len(cats) > 8 else 0, ha="right" if len(cats) > 8 else "center", fontsize=8)
            if empile:
                bas = bas + np.nan_to_num(vals)
        if ligne_y is not None:
            ax.axhline(ligne_y, color=GRIS, linestyle="--", linewidth=1)
        ax.set_title(titre, loc="left"); ax.set_ylabel(y_titre); ax.set_xlabel(x_titre); ax.grid(alpha=0.3, axis="x" if horizontal else "y")
        if len(series) > 1:
            ax.legend(fontsize=8, frameon=False)
        plt.tight_layout(); plt.show()


def g_histos(blocs, titre_general="", hauteur=320):
    """Plusieurs histogrammes côte à côte : blocs = [(titre, valeurs, libellé x)]."""
    n_ = len(blocs)
    if PLOTLY:
        try:
            from plotly.subplots import make_subplots
        except Exception:
            from plotly.tools import make_subplots
        fig = make_subplots(rows=1, cols=n_, subplot_titles=[b[0] for b in blocs])
        for i, b in enumerate(blocs):
            fig.add_trace(go.Histogram(x=list(pd.Series(b[1]).dropna()), nbinsx=40, marker=dict(color=BLEU), showlegend=False,
                                       hovertemplate=b[2] + " : %{x}<br>GAB : %{y}<extra></extra>"), row=1, col=i + 1)
            fig.layout["xaxis{}".format(i + 1 if i else "")].update(title=b[2], gridcolor="#eaeef2")
            fig.layout["yaxis{}".format(i + 1 if i else "")].update(title="nombre de GAB" if i == 0 else "", gridcolor="#eaeef2")
        fig.layout.update(height=hauteur, paper_bgcolor="#fff", plot_bgcolor="#fff", margin=dict(l=50, r=20, t=60, b=45), bargap=0.05,
                          font=dict(family="Segoe UI, Helvetica, Arial, sans-serif", size=12, color=NOIR), title=dict(text=titre_general, x=0.01))
        montrer(fig)
    else:
        fig, axes = plt.subplots(1, n_, figsize=(4.4 * n_, hauteur / 80.0))
        for ax, b in zip(np.atleast_1d(axes), blocs):
            ax.hist(pd.Series(b[1]).dropna(), bins=40, color=BLEU); ax.set_title(b[0], fontsize=10); ax.set_xlabel(b[2]); ax.grid(alpha=0.3)
        np.atleast_1d(axes)[0].set_ylabel("nombre de GAB")
        plt.tight_layout(); plt.show()


FRANCE = [(2.54, 51.09), (3.15, 50.78), (3.65, 50.48), (4.15, 50.25), (4.8, 50.15), (4.85, 49.8), (5.5, 49.55), (6.15, 49.5), (6.75, 49.15), (7.6, 49.05), (8.2, 48.95),
          (7.8, 48.5), (7.55, 47.85), (7.55, 47.55), (6.95, 47.5), (6.15, 46.95), (6.05, 46.4), (6.8, 46.4), (6.95, 45.95), (7.15, 45.85), (6.8, 45.6), (7.15, 45.25),
          (6.65, 45.0), (7.0, 44.7), (6.85, 44.3), (7.5, 43.85), (7.0, 43.55), (6.6, 43.15), (5.9, 43.1), (5.2, 43.35), (4.6, 43.4), (3.9, 43.5), (3.25, 43.25),
          (3.05, 42.9), (3.15, 42.45), (2.55, 42.35), (1.75, 42.55), (1.35, 42.7), (0.65, 42.7), (-0.3, 42.8), (-1.0, 43.05), (-1.75, 43.35), (-1.5, 43.6), (-1.25, 44.4),
          (-1.2, 45.0), (-1.05, 45.6), (-1.2, 46.15), (-2.1, 46.8), (-2.2, 47.15), (-2.5, 47.3), (-3.1, 47.5), (-4.15, 47.8), (-4.4, 47.95), (-4.75, 48.35), (-4.5, 48.65),
          (-3.6, 48.75), (-2.8, 48.55), (-2.0, 48.65), (-1.6, 48.65), (-1.6, 49.2), (-1.9, 49.7), (-1.25, 49.7), (-1.1, 49.35), (0.1, 49.45), (0.3, 49.75), (1.4, 50.1),
          (1.6, 50.4), (1.6, 50.85), (2.54, 51.09)]
CORSE = [(9.4, 43.0), (9.55, 42.6), (9.4, 41.9), (9.25, 41.4), (8.8, 41.55), (8.6, 41.9), (8.55, 42.35), (8.75, 42.6), (9.3, 42.7), (9.4, 43.0)]


def g_points(x, y, titre, textes=None, couleurs=None, tailles=None, x_titre="", y_titre="", hauteur=520, groupes=None, aspect=None, axes=True):
    """Nuage de points avec infobulle (textes). groupes = [(nom, x, y, textes, couleur, taille)] tracés par-dessus un fond gris."""
    if PLOTLY:
        data = [] if not len(list(x)) else [go.Scattergl(x=list(x), y=list(y), mode="markers", name="tous les GAB", text=list(textes) if textes is not None else None,
                             marker=dict(color=couleurs if couleurs is not None else GRIS_CLAIR, size=tailles if tailles is not None else 4, opacity=0.85,
                                         colorscale="Blues" if couleurs is not None and not isinstance(couleurs, str) else None,
                                         showscale=couleurs is not None and not isinstance(couleurs, str), colorbar=dict(title="", thickness=10, len=0.5)),
                             hovertemplate="%{text}<extra></extra>" if textes is not None else None)]
        carte = bool(aspect) and not axes
        if carte:                    # une carte : le contour de la France en fond, sous les points
            data = [go.Scatter(x=[p_[0] for p_ in poly], y=[p_[1] for p_ in poly], mode="lines", fill="toself", fillcolor="#f3f5f8", line=dict(color="#b9c2cc", width=1),
                               hoverinfo="skip", showlegend=False) for poly in (FRANCE, CORSE)] + data
        for gnom, gx, gy, gt, gc, gs in (groupes or []):
            data.append(go.Scattergl(x=list(gx), y=list(gy), mode="markers", name=gnom, text=list(gt), hovertemplate="%{text}<extra></extra>",
                                     marker=dict(color=gc, size=gs, line=dict(color="#fff", width=1))))
        lay = _layout(titre, x_titre, y_titre, hauteur, showlegend=bool(groupes))
        if aspect:
            lay.yaxis.update(scaleanchor="x", scaleratio=aspect)
        if not axes:
            lay.xaxis.update(visible=False)
            lay.yaxis.update(visible=False)
        if carte:
            lay.xaxis.update(range=[-5.3, 9.8])
            lay.yaxis.update(range=[41.2, 51.3])
            lay.plot_bgcolor = "#fff"
            lay.legend.update(bgcolor="rgba(255,255,255,0.8)")
        montrer(go.Figure(data=data, layout=lay))
    else:
        fig, ax = plt.subplots(figsize=(8, hauteur / 80.0))
        if aspect and not axes:
            for poly in (FRANCE, CORSE):
                ax.fill([p_[0] for p_ in poly], [p_[1] for p_ in poly], color="#f3f5f8", edgecolor="#b9c2cc", linewidth=0.8, zorder=0)
        if len(list(x)):
            ax.scatter(x, y, s=5, c=couleurs if couleurs is not None else GRIS_CLAIR, cmap="Blues" if couleurs is not None and not isinstance(couleurs, str) else None, alpha=0.8)
        for gnom, gx, gy, gt, gc, gs in (groupes or []):
            ax.scatter(gx, gy, s=gs * 6, c=gc, label=gnom, edgecolors="#fff", zorder=3)
        if aspect:
            ax.set_aspect(aspect)
        if not axes:
            ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(titre, loc="left"); ax.set_xlabel(x_titre); ax.set_ylabel(y_titre); ax.grid(alpha=0.3)
        if groupes:
            ax.legend(fontsize=8, frameon=False)
        plt.tight_layout(); plt.show()

## 1. Réglages
**Pour la personne qui lance le notebook.** Les lecteurs du métier peuvent passer directement à la partie A.

On y indique la fiche à lire, le nombre de GAB à lister chaque jour, les noms des familles de GAB et le nom du fichier Excel produit.

In [ ]:
DATASET_ENTREE = "fiche_gab_jour"        # la fiche GAB x jour
JOURS_A_EXAMINER = "tout"                # "tout" = toute la fiche (3 ans : compter 15 à 30 min) ; ou un nombre de jours (ex. 90)
ECARTER_FIN_INCOMPLETE = True            # écarter les derniers jours où la fiche n'a visiblement pas reçu toutes les opérations
EXCLURE_DOM_TOM = True                   # codes postaux 97x / 98x écartés
TOP_LISTE = 15                           # GAB listés chaque jour (du plus au moins atypique)
TOP_PRIORITAIRE = 10                     # les N premiers sont « prioritaires »
SEUIL_RARETE = 3.0                       # un GAB-jour est « atypique » si son écart est plus rare que 1 sur 10^3 = 1 000 (4.0 = 1 sur 10 000 : plus strict)
POIDS_SECONDAIRE = 3.0                   # les écarts sur les billets et les plages horaires, seuls, comptent 3 fois moins (rarement une urgence)
JOURS_ARRET_SIGNALE = 7                  # un GAB à l'arrêt est signalé les 7 premiers jours, puis sort du classement (« arrêt déjà signalé »)
JOUR_KPI = None                          # jour des « chiffres du jour » (section 8 bis) : None = dernier jour complet, sinon "AAAA-MM-JJ"
TOP_KPI = 3                              # nombre de GAB cités par indicateur dans les chiffres du jour
CONTAMINATION = 0.005                    # Isolation Forest : part de GAB-jours marqués « atypiques » par famille (sert aux comparaisons, pas à la liste du jour)
K_FAMILLES = 8                           # familles K-Means (8 comme dans le POC d'origine)
# Les 8 noms de familles, dans l'ordre des profils-types ci-dessous (modifier le libellé à droite seulement) :
NOMS_FAMILLES = {"nuit": "De nuit",                              # part des retraits de nuit élevée
                 "etranger": "Cartes étrangères",                # part de cartes étrangères élevée
                 "touristique": "Touristiques",                  # week-end fort et cartes étrangères
                 "tres_frequente": "Très fréquentés",            # volume et montant élevés
                 "moyen": "Fréquentation moyenne",               # volume moyen, comportement régulier
                 "peu_frequente": "Peu fréquentés",              # volume faible, jours sans retrait
                 "irregulier": "Activité irrégulière",           # retraits très variables d'un jour à l'autre
                 "standard": "Standards"}                        # proche de la moyenne sur tout
NOM_EXCEL_SORTIE = "poc_ia_detection.xlsx"  # SORTIE : nom de l'Excel généré en fin de notebook ; rien à fournir
NOM_DOSSIER_DATAIKU = None               # ex. "exports_gab" : dossier géré Dataiku où déposer l'Excel
DATASET_SORTIE = None                    # ex. "poc_ia_gab_jour" : les GAB-jours scorés (une ligne par GAB et par jour)
# Cas connus : pour VÉRIFIER, jamais pour régler. "gab" = num_automate (ou début du code site), "date" = jour de l'incident.
CAS_TESTS = [
    {"gab": "000000", "date": "2026-01-01", "quoi": "inversion de cassettes 10 € / 20 €"},   # <- numéro et date du cas confirmé
    # {"gab": "......", "date": "AAAA-MM-JJ", "quoi": "afflux de cartes UnionPay"},
    # {"gab": "......", "date": "AAAA-MM-JJ", "quoi": "GAB dégradé"},
]

# Partie A — Les données
## 2. Jusqu'à quelle date les données sont-elles fiables ?
**En bref** : avant d'analyser une journée, on vérifie qu'elle a été entièrement chargée. Les opérations arrivent progressivement : si
l'analyse tourne trop tôt, la dernière journée paraît creuse et des centaines de GAB sembleraient à tort en panne.

**Ce que vous voyez** : la date du dernier jour complet, le retard des données sur aujourd'hui, et le nombre de GAB qui ont servi chaque jour.

<details><summary><i>Comment c'est calculé (pour les curieux)</i></summary>

Pour chaque jour, on compte les GAB qui ont fait au moins un retrait et on compare à la valeur typique des 4 mêmes jours de semaine précédents. Sous 90 %, le jour est déclaré incomplet et les derniers jours incomplets sont écartés.

</details>

In [ ]:
import time
from itertools import permutations
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import RobustScaler

RESEAUX = ["interne", "cb", "visa", "mastercard", "cos", "exotique"]
EXOTIQUES = {"cup": "UnionPay", "amex": "Amex", "diners": "Diners", "jcb": "JCB"}
LIB_RESEAU = {"interne": "porteurs de la banque", "cb": "CB", "visa": "Visa", "mastercard": "Mastercard", "cos": "COS", "exotique": "étrangères"}
COLONNES = ["num_automate", "date_jour", "jour_semaine", "code_site_implantation", "ville", "code_postal", "latitude", "longitude", "statut_gab", "type_gab_e_i",
            "nb_retraits", "montant_total", "nb_inaboutis", "nb_captures", "nb_retraits_repetes_10min", "nb_heures_actives", "plus_long_trou_min",
            "nb_0h_6h", "nb_22h_24h", "nb_6h_9h", "nb_9h_12h", "nb_12h_14h", "nb_14h_18h", "nb_18h_22h",
            "nb_billets_5", "nb_billets_10", "nb_billets_20", "nb_billets_50", "nb_billets_100", "plan_cassettes",
            "nb_capture_carte_volee", "nb_capture_oubli_lecture"] + ["nb_ret_" + r for r in RESEAUX] + ["nb_ret_" + r for r in EXOTIQUES]
_ds = dataiku.Dataset(DATASET_ENTREE)
try:
    _presentes = [c["name"].lower() for c in _ds.read_schema()]
except Exception:
    _presentes = None


def lire(colonnes):
    try:
        t = _ds.get_dataframe(columns=[c for c in colonnes if c in _presentes]) if _presentes else _ds.get_dataframe()
    except Exception:
        t = _ds.get_dataframe()
    t.columns = [c.lower() for c in t.columns]
    t = t[[c for c in colonnes if c in t.columns]].copy()
    t["num_automate"] = t["num_automate"].astype(str).str.strip()
    t["date_jour"] = pd.to_datetime(t["date_jour"].astype(str).str[:10], errors="coerce")
    return t[t["date_jour"].notnull()]


t0 = time.time()
df = lire(COLONNES)
if "code_postal" in df.columns:
    df["code_postal"] = df["code_postal"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
    if EXCLURE_DOM_TOM:
        dom = df["code_postal"].str.startswith(("97", "98"))
        NB_DOM = df.loc[dom, "num_automate"].nunique()
        df = df[~dom].copy()
    else:
        NB_DOM = 0
NUM = [c for c in df.columns if c.startswith("nb_") or c in ("montant_total", "plus_long_trou_min")]
for c in NUM:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
for c in ("latitude", "longitude"):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- couverture : GAB actifs par jour face aux 4 mêmes jours de semaine précédents
FIN_FICHE, DEBUT_FICHE = df["date_jour"].max(), df["date_jour"].min()
AUJOURD_HUI = pd.Timestamp.today().normalize()
actifs_jour = df[df["nb_retraits"] > 0].groupby("date_jour")["num_automate"].nunique().reindex(pd.date_range(DEBUT_FICHE, FIN_FICHE, freq="D")).fillna(0)
ref_actifs = pd.Series({d_: np.nanmedian([actifs_jour.get(d_ - pd.Timedelta(days=7 * k_), np.nan) for k_ in range(1, 5)]) for d_ in actifs_jour.index})
taux_couverture = (actifs_jour / ref_actifs).where(ref_actifs.notnull())
jours_incomplets = taux_couverture[taux_couverture < 0.9].index
FIN, ecartes = FIN_FICHE, []
if ECARTER_FIN_INCOMPLETE:
    while FIN in jours_incomplets and len(ecartes) < 7:
        ecartes.append(FIN)
        FIN = FIN - pd.Timedelta(days=1)
df = df[df["date_jour"] <= FIN].copy()
if isinstance(JOURS_A_EXAMINER, str) and JOURS_A_EXAMINER.lower() == "tout":
    NB_JOURS_EXAMINES = max(1, (FIN - DEBUT_FICHE).days + 1 - 70)
else:
    NB_JOURS_EXAMINES = int(JOURS_A_EXAMINER)
    df = df[df["date_jour"] >= FIN - pd.Timedelta(days=NB_JOURS_EXAMINES + 70)].copy()
DEBUT_FENETRE = FIN - pd.Timedelta(days=NB_JOURS_EXAMINES - 1)
longue = df[["num_automate", "date_jour", "nb_retraits", "montant_total"] + (["nb_ret_exotique"] if "nb_ret_exotique" in df.columns else [])].copy()

# calendrier complet : un GAB sans ligne un jour = 0 retrait ce jour-là (c'est ainsi qu'on voit un GAB à l'arrêt)
df["jour_semaine"] = df["date_jour"].dt.dayofweek + 1
df = df.sort_values(["num_automate", "date_jour"]).reset_index(drop=True)
debuts = df.groupby("num_automate")["date_jour"].min()
cal = pd.concat([pd.DataFrame({"num_automate": g_, "date_jour": pd.date_range(d_, FIN, freq="D")}) for g_, d_ in debuts.items()], ignore_index=True)
if len(cal) > len(df):
    geo = [c for c in ["code_site_implantation", "ville", "code_postal", "latitude", "longitude", "statut_gab", "type_gab_e_i", "plan_cassettes"] if c in df.columns]
    df = cal.merge(df, on=["num_automate", "date_jour"], how="left")
    df[NUM] = df[NUM].fillna(0)
    df = df.sort_values(["num_automate", "date_jour"]).reset_index(drop=True)
    for c in geo:
        df[c] = df.groupby("num_automate")[c].ffill()
        df[c] = df.groupby("num_automate")[c].bfill()
    df["jour_semaine"] = df["date_jour"].dt.dayofweek + 1
if "type_gab_e_i" in df.columns:
    df["type_gab_e_i"] = df["type_gab_e_i"].fillna("Inconnu")
JOURS_FR = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
MOIS_FR = ["janv.", "févr.", "mars", "avr.", "mai", "juin", "juil.", "août", "sept.", "oct.", "nov.", "déc."]

bandeau("Les données", "fiche {} — une ligne par GAB et par jour ; lecture en {:.0f} s".format(DATASET_ENTREE, time.time() - t0))
tuiles([("GAB", nb(df["num_automate"].nunique()), "{} GAB DOM-TOM écartés".format(nb(NB_DOM)) if NB_DOM else "France métropolitaine et DOM-TOM"),
        ("Jours dans la fiche", nb((FIN_FICHE - DEBUT_FICHE).days + 1), "du {} au {}".format(DEBUT_FICHE.strftime("%d/%m/%Y"), FIN_FICHE.strftime("%d/%m/%Y"))),
        ("Dernier jour complet", FIN.strftime("%d/%m/%Y"), "{} jour(s) écarté(s) en fin de fiche".format(len(ecartes)) if ecartes else "le dernier jour de la fiche est complet", "down" if ecartes else "up"),
        ("Retard de la fiche", "{} j".format((AUJOURD_HUI - FIN).days), "entre le dernier jour complet et aujourd'hui ({})".format(AUJOURD_HUI.strftime("%d/%m/%Y"))),
        ("Période examinée", nb(NB_JOURS_EXAMINES) + " j", "du {} au {}".format(DEBUT_FENETRE.strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y"))),
        ("GAB-jours", nb(len(df)), "calendrier complet, jours à 0 compris")])
if ecartes:
    note("<b>{} journée(s) écartée(s) en fin de fiche</b> : la fiche n'y contient qu'une partie des opérations (moins de 90 % des GAB habituellement actifs). "
         "L'analyse s'arrête au <b>{}</b>, dernier jour complet.".format(len(ecartes), FIN.strftime("%d/%m/%Y")), "warn")
    display(tableau(pd.DataFrame({"jour": [jfr(d_) for d_ in sorted(ecartes)], "GAB ayant retiré": [int(actifs_jour[d_]) for d_ in sorted(ecartes)],
                                  "attendus (4 mêmes jours de semaine)": [int(round(ref_actifs[d_])) for d_ in sorted(ecartes)], "couverture": [pct(taux_couverture[d_], 0) for d_ in sorted(ecartes)]})))
_autres = [d_ for d_ in jours_incomplets if d_ <= FIN]
if _autres:
    note("{} autre(s) jour(s) incomplet(s) à l'intérieur de la période (trous de collecte) : {}. Les « GAB à l'arrêt » de ces dates sont à lire avec prudence."
         .format(len(_autres), ", ".join(d_.strftime("%d/%m/%Y") for d_ in _autres[-10:])), "warn")
_fen = actifs_jour[actifs_jour.index >= FIN_FICHE - pd.Timedelta(days=84)]
g_courbes(_fen.index, [("GAB ayant fait au moins un retrait", _fen.values, BLEU), ("attendus (4 mêmes jours de semaine)", ref_actifs.reindex(_fen.index).values, GRIS, "dash")],
          "Couverture de la fiche sur les 12 dernières semaines : combien de GAB ont retiré chaque jour", y_titre="GAB actifs", hauteur=300, y_zero=True)

## 3. Ce que le modèle regarde pour chaque GAB, chaque jour
**En bref** : deux types d'informations. **Le profil du jour** : montant moyen, part des retraits de nuit, part de cartes étrangères, part de
billets de 10 €, retraits échoués, cartes avalées. **L'écart à l'habitude du GAB** : de combien la journée s'éloigne de ce que fait
d'ordinaire ce GAB ce jour de semaine.

**À quoi ça sert** : c'est l'écart à l'habitude qui permet de voir un petit GAB qui s'arrête ou un GAB qui triple ses retraits, alors qu'en
valeur brute ils ressemblent à d'autres GAB.

<details><summary><i>Comment c'est calculé (pour les curieux)</i></summary>

Habitude = valeur typique (médiane) du même jour de semaine sur les 4 semaines précédentes, sinon des 28 jours précédents. Écart = distance à l'habitude divisée par la variation habituelle du GAB : 3 veut dire « trois fois plus loin que d'ordinaire ». Un GAB fermé tous les dimanches n'est pas compté à l'arrêt le dimanche ; les jours fériés de fermeture sont reconnus par les données.

</details>

In [ ]:
# --- qualité
manquants = df.isnull().sum()
manquants = manquants[manquants > 0]
incoh = {"retraits négatifs": int((df["nb_retraits"] < 0).sum()), "montant > 0 sans retrait": int(((df["montant_total"] > 0) & (df["nb_retraits"] == 0)).sum()),
         "retraits sans montant": int(((df["montant_total"] == 0) & (df["nb_retraits"] > 0)).sum())}
df["nb_retraits"] = df["nb_retraits"].clip(lower=0)
tuiles([("Colonnes avec des valeurs manquantes", nb(len(manquants)), ", ".join(manquants.index[:4]) if len(manquants) else "aucune")] +
       [(k, nb(v), "GAB-jours") for k, v in incoh.items()])

# --- profil du jour
ret = df["nb_retraits"].clip(lower=1)
df["montant_moyen"] = np.where(df["nb_retraits"] > 0, df["montant_total"] / ret, np.nan)
df["nb_nuit"] = df["nb_0h_6h"] + (df["nb_22h_24h"] if "nb_22h_24h" in df.columns else 0)
df["pct_nuit"] = 100 * df["nb_nuit"] / ret
for r in RESEAUX:
    if "nb_ret_" + r in df.columns:
        df["pct_" + r] = 100 * df["nb_ret_" + r] / ret
df["pct_etranger"] = df["pct_exotique"] if "pct_exotique" in df.columns else 0.0
df["nb_exotique"] = df["nb_ret_exotique"] if "nb_ret_exotique" in df.columns else sum(df.get("nb_ret_" + r, 0) for r in EXOTIQUES)
bil = df[[c for c in ["nb_billets_5", "nb_billets_10", "nb_billets_20", "nb_billets_50", "nb_billets_100"] if c in df.columns]].sum(axis=1)
for v in (10, 20, 50):
    df["part_{}".format(v)] = np.where(bil > 0, 100 * df["nb_billets_{}".format(v)] / bil.clip(lower=1), np.nan)
df["taux_inaboutis"] = 1000 * df["nb_inaboutis"] / ret
df["taux_captures"] = 1000 * df["nb_captures"] / ret
df["taux_repetes"] = 1000 * df["nb_retraits_repetes_10min"] / ret
df["pct_matin"] = 100 * (df["nb_6h_9h"] + df["nb_9h_12h"]) / ret if "nb_9h_12h" in df.columns else np.nan
df["pct_soir"] = 100 * df["nb_18h_22h"] / ret if "nb_18h_22h" in df.columns else np.nan
df["plan"] = df["plan_cassettes"].astype(str) if "plan_cassettes" in df.columns else "?"
PROFIL = [c for c in ["montant_moyen", "pct_nuit", "pct_etranger", "pct_interne", "pct_cos", "taux_inaboutis", "taux_captures", "taux_repetes", "part_10", "part_20", "part_50", "pct_matin", "pct_soir"] if c in df.columns and df[c].notnull().any()]
LIB_VAR = {"montant_moyen": "montant moyen (€)", "pct_nuit": "% retraits de nuit", "pct_etranger": "% cartes étrangères", "pct_interne": "% porteurs de la banque", "pct_cos": "% COS",
           "taux_inaboutis": "inaboutis pour 1 000", "taux_captures": "captures pour 1 000", "taux_repetes": "retraits refaits pour 1 000", "part_10": "% billets de 10 €", "part_20": "% billets de 20 €",
           "part_50": "% billets de 50 €", "pct_matin": "% retraits le matin", "pct_soir": "% retraits le soir", "nb_retraits": "retraits du jour",
           "z_retraits": "écart retraits", "z_montant_moyen": "écart montant moyen", "z_repetes": "écart retraits refaits", "z_inaboutis": "écart inaboutis", "z_captures": "écart captures",
           "z_exotique": "écart cartes étrangères", "z_nuit": "écart nuit", "z_part_10": "écart % 10 €", "z_part_20": "écart % 20 €", "z_part_50": "écart % 50 €", "z_heures_actives": "écart heures actives"}

# --- écarts à l'habitude (vectorisé : indispensable sur 3 ans)
t0 = time.time()
INDIC = {"retraits": ("nb_retraits", "compte", 1.0), "montant_moyen": ("montant_moyen", "ratio", 0.05), "repetes": ("nb_retraits_repetes_10min", "compte", 0.8),
         "inaboutis": ("nb_inaboutis", "compte", 0.8), "captures": ("nb_captures", "compte", 0.8), "exotique": ("nb_exotique", "compte", 0.8), "nuit": ("nb_nuit", "compte", 0.8),
         "part_10": ("part_10", "part", 2.0), "part_20": ("part_20", "part", 2.0), "part_50": ("part_50", "part", 2.0), "heures_actives": ("nb_heures_actives", "compte", 0.8)}
INDIC = {k: v for k, v in INDIC.items() if v[0] in df.columns}
actif = df["nb_retraits"] > 0


def glisse(valeur, cles, fenetre, mini, fonction="median", decalage=1):
    cles = [df[c] if isinstance(c, str) else c for c in cles]
    s_ = valeur.groupby(cles).shift(decalage) if decalage else valeur
    r_ = getattr(s_.groupby(cles).rolling(fenetre, min_periods=mini), fonction)()
    return r_.reset_index(level=list(range(len(cles))), drop=True).reindex(valeur.index)


def habitude(valeur, cles):
    return glisse(valeur, cles + ["jour_semaine"], 4, 3).fillna(glisse(valeur, cles, 28, 10))


for nom, (col, typ, plancher) in INDIC.items():
    cles = ["num_automate", "plan"] if typ == "part" else ["num_automate"]
    df["hab_" + nom] = habitude(df[col].where(actif), cles)
    if typ == "compte":
        ecart = 2 * np.sqrt(df[col] + 0.375) - 2 * np.sqrt(df["hab_" + nom] + 0.375)
    elif typ == "ratio":
        ecart = np.log(df[col] / df["hab_" + nom])
    else:
        ecart = df[col] - df["hab_" + nom]
    variation = glisse(ecart.where(actif).abs(), cles, 28, 10) * 1.4826
    zc = (ecart / variation.clip(lower=plancher)).clip(-25, 25)
    df["z_" + nom] = (zc - zc.where(actif).groupby(df["date_jour"]).transform("median")).clip(-25, 25)   # effet du jour mesuré sur les GAB actifs seulement
Z = ["z_" + k for k in INDIC]
# --- Silences. Un GAB sans retrait n'est « à l'arrêt » que s'il est normalement en service ce jour de semaine.
# (a) part de jours sans retrait de ce GAB, ce jour de semaine, sur les 8 semaines précédentes : un GAB fermé tous les dimanches n'est pas en panne le dimanche
df["part_zero_habituelle"] = glisse((~actif).astype(float), ["num_automate", "jour_semaine"], 8, 3, "mean")
# (b) jours de fermeture collective (jours fériés), reconnus par les données : la majorité des GAB fermés le dimanche sont aussi à 0 ce jour-là
_dim = df[df["jour_semaine"] == 7].assign(_z=(df.loc[df["jour_semaine"] == 7, "nb_retraits"] == 0).astype(float)).groupby("num_automate")["_z"].mean()
df["ferme_le_dimanche"] = (df["num_automate"].map(_dim).fillna(0) >= 0.7).values
_pz = (~actif)[df["ferme_le_dimanche"]].groupby(df.loc[df["ferme_le_dimanche"], "date_jour"]).mean()
JOURS_FERMETURE_COLLECTIVE = sorted(d_ for d_ in _pz[_pz >= 0.5].index if d_.dayofweek != 6)
_collectif = df["date_jour"].isin(JOURS_FERMETURE_COLLECTIVE) & df["ferme_le_dimanche"]
# (c) nombre de jours consécutifs sans retrait
_bloc = (actif | (df["num_automate"] != df["num_automate"].shift())).cumsum()
df["jours_a_zero"] = (~actif).astype(int).groupby(_bloc).cumsum()
df["a_l_arret"] = ~actif & (df["hab_retraits"] >= 10) & (df["part_zero_habituelle"].fillna(0) <= 0.15) & ~_collectif
df["arret_signale"] = df["a_l_arret"] & (df["jours_a_zero"] <= JOURS_ARRET_SIGNALE)
# un jour sans retrait qui n'est pas un arrêt à signaler (fermeture habituelle, jour férié, arrêt déjà signalé) ne porte aucun écart
_neutre = ~actif & ~df["arret_signale"]
df.loc[_neutre, Z] = 0.0
df.loc[df["arret_signale"], [z_ for z_ in Z if z_ != "z_retraits"]] = 0.0
df.loc[df["arret_signale"], "z_retraits"] = -(2 * np.sqrt(df.loc[df["arret_signale"], "hab_retraits"] + 0.375)).clip(upper=25)
df["historique_ok"] = df["hab_retraits"].notnull()
dfd = df[(df["date_jour"] >= DEBUT_FENETRE) & df["historique_ok"]].copy().reset_index(drop=True)
dfd["flag_statistique"] = (dfd[Z].abs() >= 4).any(axis=1).astype(int)      # au moins un écart extrême : l'équivalent du « flag SQL » du POC
tuiles([("Variables du profil du jour", str(len(PROFIL)), ", ".join(LIB_VAR.get(c, c) for c in PROFIL[:5]) + "…"),
        ("Écarts à l'habitude", str(len(Z)), ", ".join(LIB_VAR.get(c, c).replace("écart ", "") for c in Z[:6]) + "…"),
        ("Habitude calculable", pct(df.loc[actif & (df["date_jour"] >= DEBUT_FENETRE), "historique_ok"].mean(), 1), "des GAB-jours actifs de la période"),
        ("GAB-jours examinés", nb(len(dfd)), "{} GAB sur {} jours".format(nb(dfd["num_automate"].nunique()), nb(dfd["date_jour"].nunique()))),
        ("Au moins un écart extrême", nb(dfd["flag_statistique"].sum()), "{} des GAB-jours (|écart| ≥ 4) ; {} GAB distincts".format(pct(dfd["flag_statistique"].mean(), 2), nb(dfd.loc[dfd["flag_statistique"] == 1, "num_automate"].nunique()))),
        ("GAB fermés le dimanche", nb(df.loc[df["ferme_le_dimanche"], "num_automate"].nunique()), "à 0 au moins 7 dimanches sur 10 : pas en panne le dimanche"),
        ("GAB-jours à l'arrêt", nb(dfd["a_l_arret"].sum()), "dont {} signalés (les {} premiers jours) ; {} GAB distincts".format(nb(dfd["arret_signale"].sum()), JOURS_ARRET_SIGNALE, nb(dfd.loc[dfd["a_l_arret"], "num_automate"].nunique()))),
        ("Temps de calcul", "{:.0f} s".format(time.time() - t0), "habitudes et écarts")])
if JOURS_FERMETURE_COLLECTIVE:
    note("Jours de fermeture collective reconnus par les données (les GAB fermés le dimanche sont aussi à 0 ces jours-là, donc pas d'alerte) : {}.".format(
        ", ".join(jfr(d_) for d_ in JOURS_FERMETURE_COLLECTIVE[-12:])))
retenir(["Un GAB sans retrait n'est « à l'arrêt » que s'il est normalement en service ce jour de semaine : <b>{}</b> GAB sont fermés tous les dimanches, et les jours de fermeture collective sont reconnus par les données.".format(nb(df.loc[df["ferme_le_dimanche"], "num_automate"].nunique())),
         "Un arrêt est signalé les <b>{}</b> premiers jours, puis sort du classement (« arrêt déjà signalé ») pour laisser la place aux nouveaux incidents.".format(JOURS_ARRET_SIGNALE)])
# exemple : un GAB de volume médian, ses retraits face à son habitude
_cand = dfd[dfd["nb_retraits"] > 0].groupby("num_automate")["nb_retraits"].median()
_ex = (_cand - _cand.median()).abs().idxmin()
_s = df[(df["num_automate"] == _ex) & (df["date_jour"] > FIN - pd.Timedelta(days=112))]
g_courbes(_s["date_jour"], [("retraits du jour", _s["nb_retraits"], BLEU), ("habitude (même jour de semaine, 4 semaines)", _s["hab_retraits"], GRIS, "dash")],
          "Exemple : GAB {} (volume médian) — ses retraits face à son habitude, 16 dernières semaines".format(_ex), y_titre="retraits", hauteur=300)

## 4. Journées ordinaires et journées suspectes : à quoi ressemblent-elles ?
**Ce que vous voyez** : pour chaque indicateur, les journées ordinaires (bleu) et celles qui ont au moins un écart très fort (orange). Quand
les deux formes se superposent, l'indicateur brut ne distingue rien : c'est bien l'écart à l'habitude qui porte le signal. La carte de
couleurs vérifie que les indicateurs ne disent pas tous la même chose (rouge = liés, blanc = indépendants).

In [ ]:
def g_histo2(blocs, titre_general="", hauteur=620, colonnes=3):
    """Histogrammes superposés : blocs = [(titre, valeurs ordinaires, valeurs suspectes)]."""
    lignes_ = int(np.ceil(len(blocs) / float(colonnes)))
    if PLOTLY:
        try:
            from plotly.subplots import make_subplots
        except Exception:
            from plotly.tools import make_subplots
        fig = make_subplots(rows=lignes_, cols=colonnes, subplot_titles=[b[0] for b in blocs])
        for i, (t_, a_, b_) in enumerate(blocs):
            r_, c_ = i // colonnes + 1, i % colonnes + 1
            fig.add_trace(go.Histogram(x=list(a_), nbinsx=40, histnorm="probability", marker=dict(color=BLEU), opacity=0.6, name="ordinaires", showlegend=(i == 0), legendgroup="a"), row=r_, col=c_)
            if len(b_):
                fig.add_trace(go.Histogram(x=list(b_), nbinsx=25, histnorm="probability", marker=dict(color="#e36209"), opacity=0.6, name="avec un écart extrême", showlegend=(i == 0), legendgroup="b"), row=r_, col=c_)
        fig.layout.update(height=hauteur, barmode="overlay", paper_bgcolor="#fff", plot_bgcolor="#fff", margin=dict(l=40, r=20, t=70, b=40), bargap=0.05,
                          font=dict(family="Segoe UI, Helvetica, Arial, sans-serif", size=11, color=NOIR), title=dict(text=titre_general, x=0.01),
                          legend=dict(orientation="h", y=1.08, x=1, xanchor="right"))
        for k_ in range(1, len(blocs) + 1):
            fig.layout["xaxis{}".format(k_ if k_ > 1 else "")].update(gridcolor="#eaeef2")
            fig.layout["yaxis{}".format(k_ if k_ > 1 else "")].update(gridcolor="#eaeef2", showticklabels=False)
        montrer(fig)
    else:
        fig, axes = plt.subplots(lignes_, colonnes, figsize=(5 * colonnes, 3.2 * lignes_))
        for ax, (t_, a_, b_) in zip(np.atleast_1d(axes).flatten(), blocs):
            ax.hist(a_, bins=40, color=BLEU, alpha=0.6, density=True, label="ordinaires")
            if len(b_):
                ax.hist(b_, bins=25, color="#e36209", alpha=0.6, density=True, label="avec un écart extrême")
            ax.set_title(t_, fontsize=10); ax.grid(alpha=0.3)
        np.atleast_1d(axes).flatten()[0].legend(fontsize=8)
        plt.suptitle(titre_general); plt.tight_layout(); plt.show()


def g_chaleur(mat, titre, hauteur=560):
    """Matrice (DataFrame carrée) en carte de chaleur, −1 à 1."""
    labs = [LIB_VAR.get(c, c) for c in mat.columns]
    if PLOTLY:
        fig = go.Figure(data=[go.Heatmap(z=mat.values, x=labs, y=labs, zmin=-1, zmax=1, colorscale="RdBu", reversescale=True, colorbar=dict(thickness=10),
                                         hovertemplate="%{y} × %{x}<br>corrélation : %{z:.2f}<extra></extra>")],
                        layout=_layout(titre, hauteur=hauteur, margin=dict(l=160, r=20, t=50, b=160)))
        fig.layout.xaxis.update(tickangle=-60, tickfont=dict(size=10), showgrid=False)
        fig.layout.yaxis.update(tickfont=dict(size=10), showgrid=False, autorange="reversed")
        montrer(fig)
    else:
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-1, vmax=1)
        ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs))); ax.set_xticklabels(labs, rotation=90, fontsize=7); ax.set_yticklabels(labs, fontsize=7)
        plt.colorbar(im, ax=ax, shrink=0.7); ax.set_title(titre, loc="left"); plt.tight_layout(); plt.show()


ech = dfd.sample(min(60 * 1000, len(dfd)), random_state=0)
variables = [("nb_retraits", "Retraits du jour"), ("montant_moyen", "Montant moyen (€)"), ("taux_captures", "Captures pour 1 000 retraits"), ("pct_nuit", "% de retraits de nuit"),
             ("pct_etranger", "% de cartes étrangères"), ("taux_repetes", "Retraits refaits < 10 min pour 1 000")]
blocs = []
for col, t_ in variables:
    a_, b_ = ech.loc[ech["flag_statistique"] == 0, col].dropna(), ech.loc[ech["flag_statistique"] == 1, col].dropna()
    haut = np.nanpercentile(pd.concat([a_, b_]), 99) if len(a_) else 1
    blocs.append((t_, a_.clip(upper=haut), b_.clip(upper=haut)))
g_histo2(blocs, "Répartition des GAB-jours ordinaires (bleu) et de ceux qui ont un écart extrême (orange) — échantillon de {} GAB-jours".format(nb(len(ech))))
if "type_gab_e_i" in dfd.columns:
    _t = ech.groupby("type_gab_e_i").agg(**{"GAB-jours": ("nb_retraits", "size"), "retraits (médiane)": ("nb_retraits", "median"), "montant moyen (médiane)": ("montant_moyen", "median"),
                                           "% nuit (médiane)": ("pct_nuit", "median"), "% cartes étrangères (moyenne)": ("pct_etranger", "mean"), "captures pour 1 000 (moyenne)": ("taux_captures", "mean")}).reset_index().rename(columns={"type_gab_e_i": "type (I = intérieur, E = extérieur)"})
    display(tableau(_t, "Intérieur / extérieur : des comportements différents", dec=1))
g_chaleur(ech[PROFIL + Z].corr(method="spearman"), "Les indicateurs disent-ils la même chose ? (rouge = liés, blanc = indépendants)")
retenir(["<b>{}</b> des GAB-jours ont au moins un écart extrême ; en valeur brute (volume, montant), ils ressemblent aux autres : seule la comparaison à l'habitude du GAB les distingue.".format(pct(dfd["flag_statistique"].mean(), 2)),
         "Les écarts sont peu corrélés entre eux (hors parts de billets, liées par construction) : chacun apporte une information différente au modèle."])

### Les cartes étrangères : qui, combien, quand ?
Pour chaque réseau étranger (UnionPay, Amex, Diners, JCB), les 15 GAB qui en voient le plus. En orange, les GAB qui ont connu au moins une
journée à 5 retraits ou plus de ce réseau : c'est le seuil d'une « rafale ».

In [ ]:
reseaux_def = [("nb_ret_" + r, lib) for r, lib in EXOTIQUES.items() if "nb_ret_" + r in dfd.columns]
for col, lib in reseaux_def:
    par_gab = dfd.groupby("num_automate")[col].mean()
    top_ = par_gab.sort_values(ascending=False).head(15)
    maxj = dfd.groupby("num_automate")[col].max().reindex(top_.index)
    g_barres(list(top_.index[::-1]), [("retraits par jour (moyenne)", top_.values[::-1].round(2), ["#e36209" if m_ >= 5 else BLEU for m_ in maxj.values[::-1]])],
             "{} : les 15 GAB les plus concernés — parc : {} retrait par GAB et par jour, {} GAB-jours à 5 ou plus ({} GAB)".format(
                 lib, nb(par_gab.mean(), 3), nb((dfd[col] >= 5).sum()), nb(dfd.loc[dfd[col] >= 5, "num_automate"].nunique())), hauteur=360, horizontal=True)
if "nb_ret_exotique" in longue.columns:
    mens = longue.assign(mois=longue["date_jour"].dt.to_period("M").dt.to_timestamp()).groupby("mois").agg(exotique=("nb_ret_exotique", "sum"), retraits=("nb_retraits", "sum"), gab=("num_automate", "nunique"))
    g_courbes(mens.index, [("cartes étrangères par GAB et par mois", (mens["exotique"] / mens["gab"]).round(2), "#e36209")], "Évolution mensuelle des cartes étrangères (par GAB)", hauteur=280,
              hover="%{x|%m/%Y}<br>%{y:.2f} par GAB<extra></extra>")
    g_courbes(mens.index, [("retraits par GAB et par mois", (mens["retraits"] / mens["gab"]).round(0), BLEU)], "Évolution mensuelle des retraits (par GAB)", hauteur=280,
              hover="%{x|%m/%Y}<br>%{y:,.0f} par GAB<extra></extra>")

## 5. Les rythmes du parc
**Indice** : 1 = un jour ordinaire, 1,2 = 20 % de retraits en plus. Ces rythmes (jour de semaine, saison) sont neutralisés avant de juger un GAB.

In [ ]:
parc = longue.groupby("date_jour").agg(retraits=("nb_retraits", "sum"), gab_actifs=("nb_retraits", lambda s: int((s > 0).sum())))
g_courbes(parc.index, [("retraits du parc", parc["retraits"], BLEU_CLAIR), ("moyenne glissante 28 jours", parc["retraits"].rolling(28, center=True, min_periods=10).mean(), BLEU)],
          "Retraits du parc, jour par jour", y_titre="retraits", hauteur=330)
g_courbes(parc.index, [("GAB ayant fait au moins un retrait", parc["gab_actifs"], BLEU)], "Taille du parc actif, jour par jour", y_titre="GAB actifs", hauteur=240, y_zero=True)
jds = longue.groupby(longue["date_jour"].dt.dayofweek)["nb_retraits"].sum() / longue.groupby(longue["date_jour"].dt.dayofweek)["date_jour"].nunique()
mois_i = longue.groupby(longue["date_jour"].dt.to_period("M"))["nb_retraits"].sum() / longue.groupby(longue["date_jour"].dt.to_period("M"))["date_jour"].nunique()
g_barres([JOURS_FR[d] for d in jds.index], [("indice", (jds / jds.mean()).round(2).values, BLEU)], "Effet du jour de semaine (1 = jour moyen)", y_titre="indice", hauteur=260, ligne_y=1)
g_courbes(mois_i.index.to_timestamp() + pd.Timedelta(days=14), [("indice du mois", (mois_i / mois_i.mean()).round(2).values, BLEU)], "Effet du mois (1 = jour moyen)", y_titre="indice", hauteur=260,
          hover="%{x|%m/%Y}<br>indice : %{y:.2f}<extra></extra>")

# Partie B — Les familles de GAB
## 6. Quels types de GAB ? Les familles
**En bref** : le modèle range les GAB en 8 familles de comportement (de nuit, cartes étrangères, touristiques, très fréquentés, fréquentation
moyenne, peu fréquentés, activité irrégulière, standards).

**À quoi ça sert** : chaque GAB est ensuite jugé **par rapport à ses pareils**, pas au parc entier.

**Ce que vous voyez** : comment le nombre de familles a été choisi, le profil de chaque famille, et deux cartes (une vue d'ensemble et la
carte de France) où chaque point est un GAB coloré par sa famille.

<details><summary><i>Comment c'est calculé (pour les curieux)</i></summary>

K-Means sur le profil de chaque GAB (volume, montant, nuit, week-end, cartes étrangères, incidents, irrégularité, jours à zéro). Les noms sont attribués par ressemblance à huit profils types, chaque nom servant une seule fois. Ils se changent en section 1.

</details>

In [ ]:
gp = dfd.groupby("num_automate")
we = dfd[dfd["jour_semaine"] >= 6].groupby("num_automate")["nb_retraits"].mean() / dfd[dfd["jour_semaine"] < 6].groupby("num_automate")["nb_retraits"].mean().replace(0, np.nan)
profil = pd.DataFrame({"retraits_par_jour": gp["nb_retraits"].mean(), "montant_moyen": gp["montant_moyen"].median(), "pct_nuit": gp["pct_nuit"].median(), "ratio_weekend": we,
                       "pct_etranger": gp["pct_etranger"].mean(), "exotique_par_jour": gp["nb_exotique"].mean(),
                       "pct_interne": gp["pct_interne"].median() if "pct_interne" in dfd.columns else 0.0, "pct_cos": gp["pct_cos"].median() if "pct_cos" in dfd.columns else 0.0,
                       "taux_captures": gp["taux_captures"].mean(), "taux_inaboutis": gp["taux_inaboutis"].mean(),
                       "cv_retraits": gp["nb_retraits"].std() / gp["nb_retraits"].mean().replace(0, np.nan),
                       "part_jours_zero": dfd.assign(_z=(dfd["nb_retraits"] == 0).astype(float)).groupby("num_automate")["_z"].mean(),
                       "heures_actives": gp["nb_heures_actives"].mean() if "nb_heures_actives" in dfd.columns else 0.0})
profil = profil.replace([np.inf, -np.inf], np.nan)
profil = profil.fillna(profil.median())
FEAT_CLUST = ["retraits_par_jour", "montant_moyen", "pct_nuit", "ratio_weekend", "pct_etranger", "pct_interne", "taux_captures", "taux_inaboutis", "cv_retraits", "part_jours_zero"]
LIB_PROFIL = {"retraits_par_jour": "retraits / jour", "montant_moyen": "montant moyen (€)", "pct_nuit": "% nuit", "ratio_weekend": "week-end / semaine", "pct_etranger": "% cartes étrangères",
              "pct_interne": "% porteurs", "taux_captures": "captures ‰", "taux_inaboutis": "inaboutis ‰", "cv_retraits": "irrégularité", "part_jours_zero": "part jours à 0", "exotique_par_jour": "cartes étrangères / jour", "heures_actives": "heures actives", "pct_cos": "% COS"}
_X = lambda t_: np.column_stack([np.log1p(t_["retraits_par_jour"])] + [t_[c] for c in FEAT_CLUST[1:]])
echelle = RobustScaler().fit(_X(profil))
Xc = np.clip(echelle.transform(_X(profil)), -5, 5)
K = list(range(2, 11))
inerties, sils = [], []
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=5, max_iter=200).fit(Xc)
    inerties.append(km.inertia_)
    sils.append(silhouette_score(Xc, km.labels_, sample_size=min(3000, len(Xc)), random_state=42))
g_courbes(K, [("dispersion dans les familles", np.round(inerties, 0), BLEU)], "Choix du nombre de familles : plus la courbe est basse, plus les GAB d'une même famille se ressemblent", x_titre="nombre de familles k", hauteur=260,
          hover="k = %{x}<br>inertie %{y:,.0f}<extra></extra>", lignes_v=[(K_FAMILLES, "k retenu")])
g_courbes(K, [("qualité de séparation", np.round(sils, 3), "#e36209")], "Qualité de séparation des familles (plus haut = familles bien distinctes)", x_titre="nombre de familles k", hauteur=260,
          hover="k = %{x}<br>silhouette %{y:.3f}<extra></extra>", lignes_v=[(K_FAMILLES, "k retenu")])
km_final = KMeans(n_clusters=K_FAMILLES, random_state=42, n_init=20, max_iter=500).fit(Xc)
labels = km_final.labels_.copy()
tailles = np.bincount(labels, minlength=K_FAMILLES)
petits = [c for c in range(K_FAMILLES) if tailles[c] < max(10, 0.01 * len(Xc))]
if petits and len(petits) < K_FAMILLES:
    grands = [c for c in range(K_FAMILLES) if c not in petits]
    m_ = np.isin(labels, petits)
    labels[m_] = np.array(grands)[((Xc[m_][:, None, :] - km_final.cluster_centers_[grands][None, :, :]) ** 2).sum(axis=2).argmin(axis=1)]
    note("{} famille(s) trop petite(s) rattachée(s) à la plus proche ({} GAB).".format(len(petits), int(m_.sum())), "warn")
profil["cluster"] = labels
pc = profil.groupby("cluster")[FEAT_CLUST].mean()
pz = (pc - pc.mean()) / pc.std(ddof=0).replace(0, 1)


def score_famille(r, role):
    v = lambda c: float(r[c]) if c in r.index else 0.0
    vol, mont, nuit, we_, etr, cv, zero, cap, interne = v("retraits_par_jour"), v("montant_moyen"), v("pct_nuit"), v("ratio_weekend"), v("pct_etranger"), v("cv_retraits"), v("part_jours_zero"), v("taux_captures"), v("pct_interne")
    return {"nuit": 2.2 * nuit + 0.4 * we_, "etranger": 2.0 * etr - 0.4 * interne - 0.3 * vol, "touristique": 1.2 * we_ + 1.0 * etr + 0.4 * vol + 0.2 * mont,
            "tres_frequente": 1.8 * vol + 0.6 * mont - 0.3 * we_ - 0.3 * cv, "moyen": 0.8 * vol - 0.6 * abs(cv) - 0.4 * abs(nuit) - 0.3 * abs(etr) - 0.3 * zero,
            "peu_frequente": -1.7 * vol + 0.8 * zero - 0.3 * etr, "irregulier": 2.2 * cv + 0.8 * zero + 0.4 * cap,
            "standard": -0.7 * (abs(nuit) + abs(etr) + abs(cv) + abs(vol) + abs(we_))}[role]


ROLES = list(NOMS_FAMILLES)
scores = pd.DataFrame({role: [score_famille(pz.loc[c], role) for c in pz.index] for role in ROLES}, index=pz.index)
ids = list(scores.index)
meilleur, best = None, -np.inf
for perm in permutations(ROLES, len(ids)):
    tot = sum(scores.loc[c, role] for c, role in zip(ids, perm))
    if tot > best:
        best, meilleur = tot, perm
NOM_CLUSTER = {c: NOMS_FAMILLES[role] for c, role in zip(ids, meilleur)}
COULEURS_FAMILLES = dict(zip(NOMS_FAMILLES.values(), SERIE))
profil["famille"] = profil["cluster"].map(NOM_CLUSTER)
dfd["famille"] = dfd["num_automate"].map(profil["famille"])
dfd["cluster"] = dfd["num_automate"].map(profil["cluster"])
tab_fam = pc.rename(index=NOM_CLUSTER)
tab_fam.insert(0, "GAB", profil.groupby("famille").size().reindex(tab_fam.index))
tab_fam.insert(1, "% du parc", (tab_fam["GAB"] / tab_fam["GAB"].sum() * 100).round(1))
tab_fam = tab_fam.sort_values("GAB", ascending=False)
_t = tab_fam.reset_index().rename(columns=dict(LIB_PROFIL, cluster="famille"))
display(tableau(_t, "Les {} familles : profil moyen de chaque famille".format(len(tab_fam)), dec=2))
g_barres(list(tab_fam.index[::-1]), [("GAB", tab_fam["GAB"].values[::-1], [COULEURS_FAMILLES[f] for f in tab_fam.index[::-1]])], "Nombre de GAB par famille", hauteur=300, horizontal=True)
if sils[K.index(K_FAMILLES)] < 0.15:
    note("Familles peu séparées ({} sur 1) : elles se chevauchent, leurs noms sont indicatifs.".format(nb(sils[K.index(K_FAMILLES)], 2)), "warn")
Xp = PCA(n_components=2, random_state=42).fit_transform(Xc)
g_points([], [], "Vue d'ensemble des familles : chaque point est un GAB, les GAB proches se ressemblent", hauteur=520, x_titre="", y_titre="",
         groupes=[(nom, Xp[(profil["famille"] == nom).values, 0], Xp[(profil["famille"] == nom).values, 1],
                   ["GAB {}<br>{}<br>{} retraits / jour".format(i_, nom, nb(v_, 0)) for i_, v_ in zip(profil.index[(profil["famille"] == nom).values], profil.loc[profil["famille"] == nom, "retraits_par_jour"])],
                   COULEURS_FAMILLES[nom], 7) for nom in NOMS_FAMILLES.values() if (profil["famille"] == nom).any()])
ident = dfd.sort_values("date_jour").groupby("num_automate")[[c for c in ["code_site_implantation", "ville", "code_postal", "latitude", "longitude", "type_gab_e_i"] if c in dfd.columns]].last()
profil = profil.join(ident)
if "latitude" in profil.columns and profil["latitude"].notnull().any():
    _p = profil[profil["latitude"].between(41, 51.5) & profil["longitude"].between(-5.5, 10)]
    g_points([], [], "Carte des familles (survoler un point)", hauteur=600, aspect=1.4, axes=False,
             groupes=[(nom, _p.loc[_p["famille"] == nom, "longitude"], _p.loc[_p["famille"] == nom, "latitude"],
                       ["GAB {}{}<br>{}<br>{} retraits / jour".format(i_, " — " + str(v_) if "ville" in _p.columns and pd.notnull(v_) else "", nom, nb(r_, 0))
                        for i_, v_, r_ in zip(_p.index[_p["famille"] == nom], _p.loc[_p["famille"] == nom, "ville"] if "ville" in _p.columns else [None] * int((_p["famille"] == nom).sum()), _p.loc[_p["famille"] == nom, "retraits_par_jour"])],
                       COULEURS_FAMILLES[nom], 6) for nom in NOMS_FAMILLES.values() if (_p["famille"] == nom).any()])
retenir(["<b>{}</b> familles sur {} GAB ; la plus grande est <b>{}</b> ({} GAB, {}), la plus petite <b>{}</b> ({} GAB).".format(
             len(tab_fam), nb(len(profil)), tab_fam.index[0], nb(tab_fam["GAB"].iloc[0]), pct(tab_fam["GAB"].iloc[0] / tab_fam["GAB"].sum(), 0), tab_fam.index[-1], nb(tab_fam["GAB"].iloc[-1])),
         "Qualité de séparation des familles : <b>{}</b> sur 1 ({}).".format(nb(sils[K.index(K_FAMILLES)], 2), "bonne" if sils[K.index(K_FAMILLES)] >= 0.3 else ("moyenne" if sils[K.index(K_FAMILLES)] >= 0.15 else "faible : les familles se chevauchent, leurs noms sont indicatifs")),
         "Les noms décrivent ce que les données montrent (volume, nuit, cartes étrangères, irrégularité) ; ils se changent dans la section 1."])

### Les GAB qui changent de comportement
Chaque mois, chaque GAB est rattaché à la famille qui lui ressemble le plus. Un GAB qui passe **au moins 2 mois de suite** dans une autre
famille a changé de comportement : travaux, concurrence, panne récurrente, saison.

In [ ]:
dfd["mois"] = dfd["date_jour"].dt.to_period("M").dt.to_timestamp()
gm = dfd.groupby(["num_automate", "mois"])
pm = pd.DataFrame({"retraits_par_jour": gm["nb_retraits"].mean(), "montant_moyen": gm["montant_moyen"].median(), "pct_nuit": gm["pct_nuit"].median(),
                   "ratio_weekend": dfd[dfd["jour_semaine"] >= 6].groupby(["num_automate", "mois"])["nb_retraits"].mean() / dfd[dfd["jour_semaine"] < 6].groupby(["num_automate", "mois"])["nb_retraits"].mean().replace(0, np.nan),
                   "pct_etranger": gm["pct_etranger"].mean(), "pct_interne": gm["pct_interne"].median() if "pct_interne" in dfd.columns else 0.0,
                   "taux_captures": gm["taux_captures"].mean(), "taux_inaboutis": gm["taux_inaboutis"].mean(), "cv_retraits": gm["nb_retraits"].std() / gm["nb_retraits"].mean().replace(0, np.nan),
                   "part_jours_zero": dfd.assign(_z=(dfd["nb_retraits"] == 0).astype(float)).groupby(["num_automate", "mois"])["_z"].mean(), "nb_jours": gm.size()})
pm = pm[pm["nb_jours"] >= 15].replace([np.inf, -np.inf], np.nan)
pm = pm.fillna(profil[FEAT_CLUST].median())
Xpm = np.clip(echelle.transform(_X(pm)), -5, 5)
pm["cluster"] = np.argmin(((Xpm[:, None, :] - km_final.cluster_centers_[None, :, :]) ** 2).sum(axis=2), axis=1)
pm["famille_mois"] = pm["cluster"].map(NOM_CLUSTER)
pm = pm.reset_index().sort_values(["num_automate", "mois"])
pm["famille_ref"] = pm["num_automate"].map(profil["famille"])
pm["hors_famille"] = (pm["famille_mois"] != pm["famille_ref"]).astype(int)
# un mois isolé hors de sa famille est du bruit ; un changement = au moins 2 mois consécutifs dans une autre famille
pm["hors_2"] = pm["hors_famille"] * pm.groupby("num_automate")["hors_famille"].shift(1).fillna(0)
pm["changement"] = ((pm["hors_2"] == 1) & (pm.groupby("num_automate")["hors_2"].shift(1).fillna(0) == 0)).astype(int)
stab = pm.groupby("num_automate").agg(mois=("mois", "size"), changements=("changement", "sum"), mois_hors_famille=("hors_famille", "sum"))
stab["etiquette"] = np.select([stab["changements"] == 0, stab["changements"] <= 1, stab["changements"] <= 3], ["Stable", "Légèrement instable", "Instable"], "Comportement atypique à surveiller")
profil = profil.join(stab[["changements", "etiquette"]])
_e = stab["etiquette"].value_counts().reindex(["Stable", "Légèrement instable", "Instable", "Comportement atypique à surveiller"]).fillna(0).astype(int)
tuiles([(k, nb(v), "GAB") for k, v in _e.items()])
_ch = pm[pm["changement"] == 1].groupby("mois").size()
if len(_ch):
    g_barres(_ch.index.strftime("%m/%Y"), [("GAB ayant changé de famille", _ch.values, BLEU)], "Changements de famille, mois par mois (nombre de GAB)", hauteur=260)
_top = stab.sort_values(["changements", "mois_hors_famille"], ascending=False).head(10).join(profil[["famille"]]).reset_index().rename(columns={"num_automate": "GAB", "mois": "mois observés", "changements": "changements de famille", "mois_hors_famille": "mois hors de sa famille", "etiquette": "étiquette", "famille": "famille actuelle"})
display(tableau(_top, "Les 10 GAB les plus instables"))

# Partie C — Le modèle et ses résultats
## 7. Comment le modèle classe les GAB chaque jour
**En bref** : chaque jour, chaque GAB reçoit une note de **rareté** : à quel point sa journée est inhabituelle par rapport à ses habitudes.
« 1 sur 1 000 » veut dire qu'une journée aussi inhabituelle n'arrive qu'une fois sur 1 000. La liste du jour reprend les GAB les plus rares,
avec les arrêts et les inversions de cassettes en tête.

**À quoi ça sert** : une liste courte et classée chaque jour (10 à 15 GAB), jamais la moitié du parc.

<details><summary><i>Comment c'est calculé (pour les curieux)</i></summary>

Pour chaque indicateur, on mesure sur toute la période la part des GAB-jours qui s'écartent autant de leur habitude. Le GAB-jour est noté par son écart le plus rare. Un modèle d'apprentissage (Isolation Forest, un par famille) repère en appoint les combinaisons de petits écarts. Testé sur un parc fictif de 5 571 GAB avec 326 anomalies cachées : ce classement en retrouve bien plus dans les 10 premiers du jour que le modèle seul.

</details>

In [ ]:
FEAT_IF = Z + ["nb_retraits"] + [c for c in ["pct_etranger", "pct_nuit", "montant_moyen", "pct_interne"] if c in PROFIL]
M = dfd[FEAT_IF].replace([np.inf, -np.inf], np.nan)
M = M.fillna(M.median())
M["nb_retraits"] = np.log1p(M["nb_retraits"])
Xm = np.clip(RobustScaler().fit_transform(M.values), -8, 8)
t0 = time.time()
mod_glob = IsolationForest(n_estimators=150, contamination="auto", random_state=42, n_jobs=8).fit(Xm)
dfd["score_global"] = -mod_glob.score_samples(Xm)
dfd["score_famille"] = np.nan
for cid, idx in dfd.groupby("cluster").groups.items():
    idx = np.asarray(idx)
    if len(idx) >= 500:
        mod = IsolationForest(n_estimators=150, contamination="auto", random_state=42, n_jobs=8).fit(Xm[idx])
        dfd.loc[idx, "score_famille"] = -mod.score_samples(Xm[idx])
    else:
        dfd.loc[idx, "score_famille"] = dfd.loc[idx, "score_global"]
dfd["score_risque"] = dfd.groupby("cluster")["score_famille"].rank(pct=True)
dfd["score_risque_global"] = dfd["score_global"].rank(pct=True)
dfd["est_anomalie_if"] = (dfd["score_risque"] >= 1 - CONTAMINATION).astype(int)
dfd["est_anomalie_global"] = (dfd["score_risque_global"] >= 1 - CONTAMINATION).astype(int)
_act = (dfd["nb_retraits"] > 0).values


def part_aussi_extreme(x, haut=True):
    """Part des GAB-jours actifs de la période dont l'écart est au moins aussi fort (vers le haut ou vers le bas)."""
    ref = np.sort(x[_act])
    if haut:
        return (len(ref) - np.searchsorted(ref, x, side="left") + 1.0) / (len(ref) + 1.0)
    return (np.searchsorted(ref, x, side="right") + 1.0) / (len(ref) + 1.0)


DEUX_SENS = ["z_retraits", "z_montant_moyen", "z_part_10", "z_part_20", "z_part_50", "z_heures_actives"]
_parts = {}
for z_ in Z:
    x_ = dfd[z_].fillna(0).values
    poids = POIDS_SECONDAIRE if z_.startswith("z_part") or z_ == "z_heures_actives" else 1.0
    _parts[z_ + " hausse"] = part_aussi_extreme(x_, True) * poids
    if z_ in DEUX_SENS:
        _parts[z_ + " baisse"] = part_aussi_extreme(x_, False) * (POIDS_SECONDAIRE if z_ == "z_montant_moyen" else poids)
if "z_part_10" in Z and "z_repetes" in Z:          # signature de l'inversion de cassettes : billets de 10 € ET retraits refaits en hausse le même jour
    _parts["inversion"] = part_aussi_extreme(np.minimum(dfd["z_part_10"].fillna(0).values, dfd["z_repetes"].fillna(0).values), True)
_parts = pd.DataFrame(_parts, index=dfd.index)
p_signal = (_parts.min(axis=1) * _parts.shape[1]).clip(upper=1.0)
p_signal[dfd["arret_signale"]] = 1e-9 / dfd.loc[dfd["arret_signale"], "hab_retraits"].clip(lower=1)      # un arrêt passe en tête, les plus gros GAB d'abord
dfd["score_signal"] = -np.log10(p_signal)
p_modele = (1 - dfd["score_risque"]) + 1.0 / len(dfd)
dfd["score_final"] = -np.log10(np.minimum(p_signal, (100 * p_modele).clip(upper=1.0)))
dfd["est_anomalie"] = (dfd["score_final"] >= SEUIL_RARETE).astype(int)
dfd["rarete"] = np.where(dfd["arret_signale"], "arrêt", np.where(dfd["score_final"] < 1, "ordinaire",
                         "1 sur " + (10 ** dfd["score_final"].clip(upper=8)).round(-1).map(lambda v_: "{:,.0f}".format(max(v_, 10)).replace(",", " "))))
dfd["rang_jour"] = dfd.groupby("date_jour")["score_final"].rank(ascending=False, method="first").astype(int)
aty = dfd["est_anomalie"] == 1
NJ = dfd["date_jour"].nunique()
bandeau("Le modèle et le classement", "{} GAB-jours × {} variables, {} familles, calculé en {:.0f} s".format(nb(len(dfd)), len(FEAT_IF), dfd["cluster"].nunique(), time.time() - t0))
tuiles([("GAB-jours atypiques", nb(aty.sum()), "écart plus rare que 1 sur {} ({} des GAB-jours)".format(nb(10 ** SEUIL_RARETE), pct(aty.mean(), 2))),
        ("GAB distincts concernés", nb(dfd.loc[aty, "num_automate"].nunique()), "sur {} GAB ; {} jamais atypiques".format(nb(dfd["num_automate"].nunique()), nb(dfd["num_automate"].nunique() - dfd.loc[aty, "num_automate"].nunique()))),
        ("Atypiques par jour", nb(aty.sum() / max(1, NJ), 1), "en moyenne, sur {} jours".format(nb(NJ))),
        ("GAB à l'arrêt signalés", nb(dfd["arret_signale"].sum()), "GAB-jours ; {} GAB distincts".format(nb(dfd.loc[dfd["arret_signale"], "num_automate"].nunique()))),
        ("Liste du jour", "{} GAB".format(TOP_LISTE), "les {} premiers prioritaires, chaque jour".format(TOP_PRIORITAIRE))])
_trois = [("est_anomalie_global", "modèle d'IA sur tout le parc (prototype d'origine)"), ("est_anomalie_if", "modèle d'IA famille par famille"), ("est_anomalie", "rareté de l'écart (retenue pour la liste du jour)")]
comp = pd.DataFrame({"méthode": [l_ for _, l_ in _trois], "GAB-jours atypiques": [int(dfd[c_].sum()) for c_, _ in _trois],
                     "GAB distincts": [dfd.loc[dfd[c_] == 1, "num_automate"].nunique() for c_, _ in _trois], "par jour en moyenne": [round(dfd[c_].sum() / NJ, 1) for c_, _ in _trois]})
display(tableau(comp, "Trois façons de compter les atypiques : le volume est comparable, la liste du jour ne retient que la dernière"))
_pf = dfd.groupby("famille").agg(**{"GAB": ("num_automate", "nunique"), "GAB-jours": ("num_automate", "size"), "atypiques (IF global)": ("est_anomalie_global", "sum"),
                                     "atypiques (IF par famille)": ("est_anomalie_if", "sum"), "atypiques (écart rare)": ("est_anomalie", "sum")})
_pf["GAB distincts atypiques (écart rare)"] = dfd[aty].groupby("famille")["num_automate"].nunique().reindex(_pf.index).fillna(0).astype(int)
_pf["part des GAB-jours atypiques"] = (_pf["atypiques (écart rare)"] / _pf["GAB-jours"]).map(lambda v: pct(v, 2))
display(tableau(_pf.reset_index(), "Par famille : le modèle global concentre les alertes sur les familles rares, l'écart rare les répartit"))
g_barres(_pf.index, [("IF global", _pf["atypiques (IF global)"].values, GRIS), ("IF par famille", _pf["atypiques (IF par famille)"].values, BLEU_CLAIR), ("écart rare (retenu)", _pf["atypiques (écart rare)"].values, BLEU)],
         "GAB-jours atypiques par famille, selon la méthode", hauteur=300)
e2 = dfd.sample(min(30000, len(dfd)), random_state=1)
g_points(e2.loc[~(e2["est_anomalie"] == 1), "z_retraits"], e2.loc[~(e2["est_anomalie"] == 1), "score_final"].clip(upper=8), "Écart de volume (retraits) et score de rareté : les atypiques (orange) ne sont pas que des écarts de volume",
         x_titre="écart des retraits à l'habitude", y_titre="score (3 = 1 sur 1 000, 4 = 1 sur 10 000)", hauteur=420, tailles=3,
         textes=["GAB {} le {}".format(a_, b_.strftime("%d/%m/%Y")) for a_, b_ in zip(e2.loc[~(e2["est_anomalie"] == 1), "num_automate"], e2.loc[~(e2["est_anomalie"] == 1), "date_jour"])],
         groupes=[("atypiques", e2.loc[e2["est_anomalie"] == 1, "z_retraits"], e2.loc[e2["est_anomalie"] == 1, "score_final"].clip(upper=8),
                   ["GAB {} le {}".format(a_, b_.strftime("%d/%m/%Y")) for a_, b_ in zip(e2.loc[e2["est_anomalie"] == 1, "num_automate"], e2.loc[e2["est_anomalie"] == 1, "date_jour"])], "#e36209", 7)])

## 8. Pourquoi chaque GAB est signalé
Chaque GAB signalé reçoit une raison en clair, avec ses chiffres du jour face à son habitude : « GAB à l'arrêt », « Inversion de cassettes
possible », « Rafale de cartes UnionPay », « Pic de cartes avalées »… Le tableau compte les raisons sur la période.

In [ ]:
def n0(x):
    return "{:,.0f}".format(x).replace(",", " ") if pd.notnull(x) else "-"


CHECKS = [("retraits", "Hausse d'activité", "Baisse d'activité", "nb_retraits", "retraits"), ("montant_moyen", "Montant moyen élevé", "Montant moyen faible", "montant_moyen", "€ par retrait"),
          ("captures", "Pic de captures de cartes", None, "nb_captures", "captures"), ("inaboutis", "Pic de retraits inaboutis", None, "nb_inaboutis", "inaboutis"),
          ("nuit", "Activité nocturne inhabituelle", None, "nb_nuit", "retraits de nuit"), ("exotique", "Afflux de cartes étrangères", None, "nb_exotique", "cartes étrangères"),
          ("repetes", "Retraits refaits par la même carte", None, "nb_retraits_repetes_10min", "retraits refaits"),
          ("part_50", "Part des billets de 50 € en hausse", "Part des billets de 50 € en baisse", "part_50", "% de billets de 50 €"),
          ("part_20", "Part des billets de 20 € en hausse", "Part des billets de 20 € en baisse", "part_20", "% de billets de 20 €"),
          ("heures_actives", "Plages horaires élargies", "Plages horaires réduites", "nb_heures_actives", "heures actives")]


def generer_raison(r):
    s = []
    inv = ("z_part_10" in r.index and "z_repetes" in r.index and pd.notnull(r["z_part_10"]) and r["z_part_10"] >= 3 and r["z_repetes"] >= 3)
    if inv:
        s.append((max(r["z_part_10"], r["z_repetes"]) + 2, "Inversion de cassettes 10 € / 20 € possible",
                  "part des billets de 10 € {} % au lieu de {} % et retraits refaits {} au lieu de {}".format(n0(r["part_10"]), n0(r["hab_part_10"]), n0(r["nb_retraits_repetes_10min"]), n0(r["hab_repetes"]))))
    if r["nb_retraits"] == 0:
        if r["arret_signale"]:
            depuis = "" if r["jours_a_zero"] <= 1 else ", depuis {} jours".format(int(r["jours_a_zero"]))
            return "GAB à l'arrêt", "aucun retrait alors qu'il en fait {} d'habitude ce jour de semaine{}".format(n0(r["hab_retraits"]), depuis)
        if r["a_l_arret"]:
            return "Arrêt déjà signalé", "sans retrait depuis {} jours (signalé les {} premiers jours)".format(int(r["jours_a_zero"]), JOURS_ARRET_SIGNALE)
        return "Fermeture habituelle", "aucun retrait, comme d'habitude ce jour-là (jour de fermeture)"
    for k, lib in EXOTIQUES.items():
        c = "nb_ret_" + k
        if c in r.index and r[c] >= 5:
            s.append((4 + r[c] / 5, "Rafale de cartes {}".format(lib), "{} retraits {} dans la journée".format(n0(r[c]), lib)))
    for k, haut_, bas_, col, unite in CHECKS:
        zk = "z_" + k
        if zk not in r.index or pd.isnull(r[zk]) or col not in r.index:
            continue
        if r[zk] >= 3 and haut_ and not (inv and k == "repetes") and r[col] > r["hab_" + k]:
            s.append((r[zk], haut_, "{} {} au lieu de {}".format(n0(r[col]), unite, n0(r["hab_" + k]))))
        elif r[zk] <= -3 and bas_ and r[col] < r["hab_" + k]:
            s.append((-r[zk], bas_, "{} {} au lieu de {}".format(n0(r[col]), unite, n0(r["hab_" + k]))))
    s.sort(key=lambda t: -t[0])
    if not s:
        petits = sorted(((abs(r["z_" + k]), k, col, unite) for k, _, _, col, unite in CHECKS if "z_" + k in r.index and pd.notnull(r["z_" + k]) and abs(r["z_" + k]) >= 2), reverse=True)[:3]
        if petits:
            return "Combinaison de petits écarts", "combinaison de petits écarts : " + " ; ".join("{} {} au lieu de {}".format(n0(r[col]), unite, n0(r["hab_" + k])) for _, k, col, unite in petits)
        return "Combinaison de petits écarts", "combinaison de petits écarts (aucun indicateur seul ne sort)"
    return s[0][1], " | ".join("{} ({})".format(a, b) for _, a, b in s[:3])


def completer_raisons(t):
    vide = t["raison_principale"] == ""
    if vide.any():
        rr = t[vide].apply(generer_raison, axis=1)
        t.loc[vide, "raison_principale"] = [a for a, _ in rr]
        t.loc[vide, "detail_atypisme"] = [b for _, b in rr]
    return t


dfd["raison_principale"], dfd["detail_atypisme"] = "", ""
haut = (dfd["score_final"] >= SEUIL_RARETE - 1) | (dfd["rang_jour"] <= TOP_LISTE)
rr = dfd[haut].apply(generer_raison, axis=1)
dfd.loc[haut, "raison_principale"] = [a for a, _ in rr]
dfd.loc[haut, "detail_atypisme"] = [b for _, b in rr]
_r = dfd[aty].groupby("raison_principale").agg(**{"GAB-jours atypiques": ("num_automate", "size"), "GAB distincts": ("num_automate", "nunique")}).sort_values("GAB-jours atypiques", ascending=False).reset_index().rename(columns={"raison_principale": "raison principale"})
display(tableau(_r, "De quoi sont faits les GAB-jours atypiques : la raison principale"))
g_barres(list(_r["raison principale"][::-1]), [("GAB-jours", _r["GAB-jours atypiques"].values[::-1], BLEU_CLAIR), ("GAB distincts", _r["GAB distincts"].values[::-1], BLEU)], "Raisons principales des GAB-jours atypiques", hauteur=max(300, 26 * len(_r)), horizontal=True)

## 8 bis. Les chiffres du jour
Pour un jour donné, les chiffres du parc comparés aux 4 mêmes jours de semaine précédents, puis **les 3 GAB en tête** pour chaque
indicateur, avec ce que ce GAB fait d'habitude. Un GAB en tête n'est pas forcément anormal : un très gros GAB fait toujours beaucoup de
retraits. La comparaison à son habitude fait la différence.

In [ ]:
kp = df[df["date_jour"] >= DEBUT_FENETRE].copy()
kp["hausse_vs_habitude"] = np.where(kp["hab_retraits"] >= 30, kp["nb_retraits"] / kp["hab_retraits"], np.nan)
kp["baisse_vs_habitude"] = np.where((kp["hab_retraits"] >= 30) & (kp["nb_retraits"] > 0), kp["hab_retraits"] / kp["nb_retraits"].clip(lower=1), np.nan)
INDICATEURS_KPI = [("nb_retraits", "hab_retraits", "Le plus de retraits", "retraits", 1, 0), ("montant_total", None, "Le plus d'argent retiré", "€", 1, 0),
                   ("nb_captures", "hab_captures", "Le plus de cartes capturées", "cartes capturées", 1, 0), ("nb_inaboutis", "hab_inaboutis", "Le plus de retraits inaboutis", "retraits inaboutis", 1, 0),
                   ("nb_exotique", "hab_exotique", "Le plus de cartes étrangères (tous réseaux)", "cartes étrangères", 1, 0)]
INDICATEURS_KPI += [("nb_ret_" + k_, None, "Le plus de cartes " + lib_, "cartes " + lib_, 1, 0) for k_, lib_ in EXOTIQUES.items() if "nb_ret_" + k_ in kp.columns]
INDICATEURS_KPI += [("nb_nuit", "hab_nuit", "Le plus de retraits de nuit (22 h – 6 h)", "retraits de nuit", 1, 0),
                    ("nb_retraits_repetes_10min", "hab_repetes", "Le plus de retraits refaits par la même carte en moins de 10 min", "retraits refaits", 1, 0),
                    ("hausse_vs_habitude", None, "Plus forte hausse par rapport à son habitude", "fois son habitude", 1.2, 1),
                    ("baisse_vs_habitude", None, "Plus forte baisse par rapport à son habitude (hors GAB à l'arrêt)", "fois moins que d'habitude", 1.2, 1)]
INDICATEURS_KPI = [i_ for i_ in INDICATEURS_KPI if i_[0] in kp.columns]
kp["famille"] = kp["num_automate"].map(profil["famille"]).fillna("")
_id = [c for c in ["ville", "code_postal", "famille"] if c in kp.columns]
morceaux_ = []
for col, hab, titre_, unite, mini, dec in INDICATEURS_KPI:
    t = kp[kp[col] >= mini].sort_values([col, "nb_retraits"], ascending=False).groupby("date_jour").head(TOP_KPI)
    t = t[["date_jour", "num_automate"] + _id + ["nb_retraits", "hab_retraits"]].assign(indicateur=titre_, valeur=kp.loc[t.index, col].round(dec if dec else 0), unite=unite,
                                                                                     habitude_du_gab=(kp.loc[t.index, hab].round(1) if hab else np.nan))
    t["rang"] = t.groupby("date_jour").cumcount() + 1
    morceaux_.append(t)
podiums = pd.concat(morceaux_, ignore_index=True)
podiums = podiums[["date_jour", "indicateur", "rang", "num_automate"] + _id + ["valeur", "unite", "habitude_du_gab", "nb_retraits", "hab_retraits"]].rename(columns={"nb_retraits": "retraits_du_jour", "hab_retraits": "retraits_habituels"})
gk = kp.groupby("date_jour")
kpi_parc = pd.DataFrame({"gab_suivis": gk.size(), "gab_avec_retraits": gk["nb_retraits"].apply(lambda v: int((v > 0).sum())), "gab_sans_retrait": gk["nb_retraits"].apply(lambda v: int((v == 0).sum())),
                         "retraits": gk["nb_retraits"].sum(), "montant_retire": gk["montant_total"].sum(), "cartes_capturees": gk["nb_captures"].sum(), "retraits_inaboutis": gk["nb_inaboutis"].sum(),
                         "cartes_etrangeres": gk["nb_exotique"].sum(), "retraits_de_nuit": gk["nb_nuit"].sum(), "retraits_refaits": gk["nb_retraits_repetes_10min"].sum()})
for k_, lib_ in EXOTIQUES.items():
    if "nb_ret_" + k_ in kp.columns:
        kpi_parc["cartes_" + k_] = gk["nb_ret_" + k_].sum()
kpi_parc["gab_a_l_arret"] = kp[kp["a_l_arret"]].groupby("date_jour").size().reindex(kpi_parc.index).fillna(0).astype(int)
kpi_parc["dont_arrets_nouveaux"] = kp[kp["a_l_arret"] & (kp["jours_a_zero"] == 1)].groupby("date_jour").size().reindex(kpi_parc.index).fillna(0).astype(int)
_tous = df.groupby("date_jour").agg(retraits=("nb_retraits", "sum"), montant_retire=("montant_total", "sum"), cartes_capturees=("nb_captures", "sum"), retraits_inaboutis=("nb_inaboutis", "sum"),
                                    cartes_etrangeres=("nb_exotique", "sum"), retraits_de_nuit=("nb_nuit", "sum"), gab_sans_retrait=("nb_retraits", lambda v: int((v == 0).sum())),
                                    gab_avec_retraits=("nb_retraits", lambda v: int((v > 0).sum())))
_ref = pd.concat([_tous.shift(7 * k_, freq="D") for k_ in range(1, 5)]).groupby(level=0).median()
for c in _tous.columns:
    kpi_parc[c + "_habituel"] = _ref[c].reindex(kpi_parc.index)
MEDAILLES = ["#d4a017", "#8a8f98", "#b0703c"]


def afficher_chiffres_du_jour(jour=None):
    """Affiche les chiffres du parc et les podiums d'un jour (None = dernier jour complet)."""
    j = pd.Timestamp(jour) if jour else kp["date_jour"].max()
    if j not in kpi_parc.index:
        note("Le {} n'est pas dans la période examinée ({} au {}).".format(j.strftime("%d/%m/%Y"), DEBUT_FENETRE.strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y")), "bad")
        return
    r = kpi_parc.loc[j]
    bandeau("Les chiffres du {}".format(jfr(j)), "parc entier ; sous chaque chiffre, l'écart avec un {} ordinaire (médiane des 4 précédents)".format(JOURS_FR[j.dayofweek]))
    items = []
    for lib_, col, unite in [("GAB avec au moins un retrait", "gab_avec_retraits", ""), ("GAB sans aucun retrait", "gab_sans_retrait", ""), ("Retraits", "retraits", ""), ("Montant retiré", "montant_retire", " €"),
                             ("Cartes capturées", "cartes_capturees", ""), ("Retraits inaboutis", "retraits_inaboutis", ""), ("Cartes étrangères", "cartes_etrangeres", ""), ("Retraits de nuit", "retraits_de_nuit", "")]:
        ref = r.get(col + "_habituel", np.nan)
        if pd.notnull(ref) and ref > 0:
            v = r[col] / ref - 1
            sous = "{}{} % face aux 4 {}s précédents ({})".format("+" if v >= 0 else "−", nb(abs(v) * 100, 0), JOURS_FR[j.dayofweek], nb(ref))
            sens = "" if abs(v) < 0.15 or ref < 10 else "down"
        else:
            sous, sens = "pas de référence", ""
        items.append((lib_, nb(r[col]) + unite, sous, sens))
    tuiles(items)
    note("Parmi les GAB sans retrait, <b>{}</b> sont à l'arrêt alors qu'ils sont normalement en service ce jour de semaine, dont <b>{}</b> depuis aujourd'hui. "
         "Les autres sont fermés comme chaque semaine ce jour-là.".format(nb(r["gab_a_l_arret"]), nb(r["dont_arrets_nouveaux"])))
    pj_ = podiums[podiums["date_jour"] == j]
    html = ['<div class="g-kpis">']
    for col, hab, titre_, unite, mini, dec in INDICATEURS_KPI:
        t = pj_[pj_["indicateur"] == titre_]
        lignes = ""
        for _, x in t.iterrows():
            lieu = " · ".join(str(x[c]) for c in ["ville", "code_postal"] if c in x.index and pd.notnull(x[c]) and str(x[c]) != "")
            if pd.notnull(x["habitude_du_gab"]):
                compar = "d'habitude : {}".format(nb(x["habitude_du_gab"], 1 if x["habitude_du_gab"] < 10 else 0))
            elif "habitude" in titre_:
                compar = "{} retraits au lieu de {}".format(nb(x["retraits_du_jour"]), nb(x["retraits_habituels"]))
            else:
                compar = "{} retraits ce jour-là".format(nb(x["retraits_du_jour"]))
            lignes += ('<tr><td style="padding:3px 6px;border:0"><span style="display:inline-block;width:18px;height:18px;border-radius:9px;background:{};color:#fff;font-size:11px;text-align:center;line-height:18px">{}</span></td>'
                       '<td style="padding:3px 6px;border:0;text-align:left"><b>{}</b><div style="font-size:11px;color:#57606a">{}</div></td>'
                       '<td style="padding:3px 6px;border:0;text-align:right;white-space:nowrap"><b>{}</b> <span style="font-size:11px;color:#57606a">{}</span><div style="font-size:11px;color:#57606a">{}</div></td></tr>'
                       ).format(MEDAILLES[min(int(x["rang"]) - 1, 2)], int(x["rang"]), x["num_automate"], lieu, nb(x["valeur"], dec), unite if dec == 0 and unite == "€" else "", compar)
        if not lignes:
            lignes = '<tr><td style="padding:6px;color:#57606a;font-size:12px;border:0">Aucun GAB concerné ce jour-là.</td></tr>'
        html.append('<div class="g-kpi" style="flex:1 1 330px;max-width:380px"><div class="l" style="text-transform:none;font-size:12.5px;font-weight:600;color:#1f2328">{}</div>'
                    '<table style="border-collapse:collapse;width:100%;font-size:12.5px">{}</table></div>'.format(titre_, lignes))
    html.append("</div>")
    display(HTML(CSS + '<div class="g-leg" style="margin-top:10px;font-weight:600;color:#1f2328">Les {} GAB en tête, indicateur par indicateur</div>'.format(TOP_KPI) + "".join(html)))


afficher_chiffres_du_jour(JOUR_KPI)
note("Pour un autre jour : <code>afficher_chiffres_du_jour(\"AAAA-MM-JJ\")</code> dans une nouvelle cellule, ou changer <code>JOUR_KPI</code> en section 1.")

## 9. Le dernier jour : les GAB à regarder en priorité
La liste quotidienne : les GAB les plus atypiques du dernier jour complet, les 10 premiers prioritaires, avec la raison et les chiffres.
Puis la carte du jour (survoler un point pour le détail) et le profil du premier GAB face à sa famille.

In [ ]:
DERNIER = dfd["date_jour"].max()
jour_d = completer_raisons(dfd[dfd["date_jour"] == DERNIER].sort_values("rang_jour").copy())
jour_d["prioritaire"] = np.where(jour_d["rang_jour"] <= TOP_PRIORITAIRE, "oui", "")
LIB_COL = {"rang_jour": "rang", "num_automate": "GAB", "ville": "ville", "famille": "famille", "rarete": "rareté de l'écart", "score_final": "score", "score_risque": "score de risque", "nb_retraits": "retraits",
           "hab_retraits": "habitude", "raison_principale": "raison principale", "detail_atypisme": "détail", "date_jour": "jour", "score_signal": "force du signal", "prioritaire": "prioritaire", "code_postal": "code postal"}
bandeau("{} — {}".format(jfr(DERNIER).capitalize(), "{} GAB à regarder".format(min(TOP_LISTE, int(((jour_d["est_anomalie"] == 1) | jour_d["arret_signale"]).sum()))) if ((jour_d["est_anomalie"] == 1) | jour_d["arret_signale"]).any() else "aucun GAB atypique"),
        "{} GAB ce jour-là ; {} atypiques (écart plus rare que 1 sur {}) ; {} à l'arrêt signalés".format(nb(len(jour_d)), nb(int(jour_d["est_anomalie"].sum())), nb(10 ** SEUIL_RARETE), nb(int(jour_d["arret_signale"].sum()))))
_t = jour_d[(jour_d["est_anomalie"] == 1) | jour_d["arret_signale"]].head(TOP_LISTE)[["rang_jour", "prioritaire", "num_automate"] + [c for c in ["ville"] if c in jour_d.columns] + ["famille", "rarete", "nb_retraits", "hab_retraits", "raison_principale", "detail_atypisme"]].copy()
_t["hab_retraits"] = _t["hab_retraits"].round(0)
if _t.empty:
    note("<b>Aucun GAB atypique ce jour-là</b> : aucune journée de GAB n'est assez inhabituelle pour entrer dans la liste. La liste reste vide plutôt que d'afficher des GAB normaux.", "ok")
else:
    display(tableau(_t, libelles=LIB_COL, longues=("détail",)))
if "latitude" in jour_d.columns and jour_d["latitude"].notnull().any():
    _p = jour_d[jour_d["latitude"].between(41, 51.5) & jour_d["longitude"].between(-5.5, 10)]
    _top = _p[(_p["rang_jour"] <= TOP_LISTE) & ((_p["est_anomalie"] == 1) | _p["arret_signale"])]
    _v = lambda r: "#{} GAB {}{}<br>{}<br>{}".format(int(r["rang_jour"]), r["num_automate"], " — " + str(r["ville"]) if "ville" in r.index and pd.notnull(r["ville"]) else "", r["raison_principale"], r["detail_atypisme"][:120])
    g_points(_p["longitude"], _p["latitude"], "Carte du {} : les {} GAB les plus atypiques (orange : prioritaires) sur le parc (gris)".format(DERNIER.strftime("%d/%m/%Y"), len(_top)),
             textes=["GAB {}{}<br>rang {}".format(a_, " — " + str(v_) if pd.notnull(v_) else "", int(r_)) for a_, v_, r_ in zip(_p["num_automate"], _p["ville"] if "ville" in _p.columns else [None] * len(_p), _p["rang_jour"])],
             couleurs=GRIS_CLAIR, tailles=4, hauteur=600, aspect=1.4, axes=False,
             groupes=[("liste du jour", _top.loc[_top["rang_jour"] > TOP_PRIORITAIRE, "longitude"], _top.loc[_top["rang_jour"] > TOP_PRIORITAIRE, "latitude"], [_v(r) for _, r in _top[_top["rang_jour"] > TOP_PRIORITAIRE].iterrows()], "#c99a06", 11),
                      ("prioritaires", _top.loc[_top["rang_jour"] <= TOP_PRIORITAIRE, "longitude"], _top.loc[_top["rang_jour"] <= TOP_PRIORITAIRE, "latitude"], [_v(r) for _, r in _top[_top["rang_jour"] <= TOP_PRIORITAIRE].iterrows()], "#e36209", 15)])
g1 = jour_d[jour_d["nb_retraits"] > 0].iloc[0] if (jour_d.head(TOP_LISTE)["nb_retraits"] > 0).any() else jour_d.iloc[0]
fam_ref = dfd[(dfd["cluster"] == g1["cluster"]) & (dfd["est_anomalie"] == 0) & (dfd["nb_retraits"] > 0)]
feats_cmp = [c for c in ["nb_retraits", "montant_moyen", "taux_captures", "taux_inaboutis", "pct_nuit", "pct_etranger", "taux_repetes", "part_10"] if c in dfd.columns]
val, ref = g1[feats_cmp].astype(float), fam_ref[feats_cmp].median()
ecarts = ((val - ref) / (ref.abs() + 1e-9) * 100).clip(-200, 500)
g_barres([LIB_VAR.get(c, c) for c in feats_cmp][::-1], [("écart à la médiane de la famille (%)", ecarts.values[::-1].round(0), ["#e36209" if v_ > 0 else BLEU for v_ in ecarts.values[::-1]])],
         "Le premier GAB actif de la liste, GAB {} (rang {}) : écart (%) à la médiane de sa famille « {} »".format(g1["num_automate"], int(g1["rang_jour"]), g1["famille"]), hauteur=320, horizontal=True, hover_suffixe=" %")
note("<b>Diagnostic du GAB {}</b> : {} — {}".format(g1["num_automate"], g1["raison_principale"], g1["detail_atypisme"]))

### Quels GAB reviennent souvent ?
Un GAB atypique un seul jour est un incident. Un GAB atypique six jours ou plus sur la période est un GAB à aller voir.

In [ ]:
gg = dfd.groupby("num_automate")
df_gab = pd.DataFrame({"jours observés": gg.size(), "jours atypiques": gg["est_anomalie"].sum(), "jours dans le top {}".format(TOP_PRIORITAIRE): gg["rang_jour"].apply(lambda s: int((s <= TOP_PRIORITAIRE).sum())),
                       "score max": gg["score_final"].max().round(2), "score de risque moyen": gg["score_risque"].mean().round(3), "retraits par jour": gg["nb_retraits"].mean().round(1),
                       "dernier jour atypique": dfd[aty].groupby("num_automate")["date_jour"].max().dt.strftime("%d/%m/%Y")})
df_gab["raison principale"] = dfd[aty].sort_values("score_final", ascending=False).groupby("num_automate")["raison_principale"].first()
df_gab["raisons"] = dfd[aty].groupby("num_automate")["raison_principale"].agg(lambda s: ", ".join("{} ({})".format(k, v) for k, v in s.value_counts().head(3).items()))
df_gab["catégorie"] = pd.cut(df_gab["jours atypiques"], bins=[-1, 0, 2, 5, 100000], labels=["jamais atypique", "1 à 2 jours", "3 à 5 jours", "6 jours ou plus"]).astype(str)
df_gab = df_gab.join(profil[["famille", "etiquette", "changements"] + [c for c in ["ville", "code_postal", "latitude", "longitude"] if c in profil.columns]]).reset_index().rename(columns={"num_automate": "GAB", "etiquette": "stabilité", "changements": "changements de famille"})
df_gab = df_gab.sort_values(["jours atypiques", "score max"], ascending=[False, False]).reset_index(drop=True)
df_gab.insert(0, "rang", np.arange(1, len(df_gab) + 1))
_c = df_gab["catégorie"].value_counts().reindex(["jamais atypique", "1 à 2 jours", "3 à 5 jours", "6 jours ou plus"]).fillna(0).astype(int)
tuiles([(k, nb(v), "GAB") for k, v in _c.items()])
g_barres(list(_c.index), [("GAB", _c.values, BLEU)], "Combien de jours chaque GAB a-t-il été atypique sur la période ? (nombre de GAB)", hauteur=260)
display(tableau(df_gab.head(TOP_LISTE)[["rang", "GAB"] + [c for c in ["ville"] if c in df_gab.columns] + ["famille", "stabilité", "jours observés", "jours atypiques", "jours dans le top {}".format(TOP_PRIORITAIRE), "score max", "raisons"]].fillna(""),
                "Les {} GAB les plus souvent atypiques sur la période : la liste à investiguer".format(TOP_LISTE), longues=("raisons",)))

## 10. Le modèle retrouve-t-il les incidents déjà connus ?
On vérifie que les incidents confirmés ressortent bien dans la liste du jour, alors qu'ils n'ont jamais servi à régler le modèle.

In [ ]:
bilan = []
for cas in CAS_TESTS:
    m_ = dfd["num_automate"] == str(cas["gab"])
    if "code_site_implantation" in dfd.columns:
        m_ = m_ | dfd["code_site_implantation"].astype(str).str.startswith(str(cas["gab"]))
    d0 = pd.Timestamp(cas["date"])
    t = dfd[m_ & (dfd["date_jour"] >= d0 - pd.Timedelta(days=3)) & (dfd["date_jour"] <= d0 + pd.Timedelta(days=2))].copy()
    bandeau("Cas {} : {}".format(cas["gab"], cas.get("quoi", "")), "incident du {}".format(d0.strftime("%d/%m/%Y")))
    if t.empty:
        note("Le GAB n'est pas dans la période autour du {} (numéro inconnu, ou période trop courte).".format(d0.strftime("%d/%m/%Y")), "bad")
        bilan.append({"cas": cas["gab"], "quoi": cas.get("quoi", ""), "résultat": "absent"})
        continue
    t = completer_raisons(t)
    _a = t[["date_jour", "famille", "rang_jour", "rarete", "score_risque", "nb_retraits", "hab_retraits", "raison_principale", "detail_atypisme"]].copy()
    _a["hab_retraits"] = _a["hab_retraits"].round(0)
    _a["score_risque"] = _a["score_risque"].round(2)
    display(tableau(_a, libelles=LIB_COL, longues=("détail",)))
    fen = t[t["date_jour"] >= d0 - pd.Timedelta(days=1)]
    ok = fen[fen["rang_jour"] <= TOP_LISTE]
    if len(ok):
        r = ok.sort_values("rang_jour").iloc[0]
        note("<b>Retrouvé.</b> Rang <b>{}</b> du jour le {} (écart : {}). Raison : {}".format(int(r["rang_jour"]), r["date_jour"].strftime("%d/%m/%Y"), r["rarete"], r["raison_principale"]), "ok")
        bilan.append({"cas": cas["gab"], "quoi": cas.get("quoi", ""), "résultat": "retrouvé", "rang du jour": int(r["rang_jour"]), "date": r["date_jour"].strftime("%d/%m/%Y"), "raison": r["raison_principale"]})
    else:
        r = fen.sort_values("rang_jour").iloc[0]
        note("<b>Non retrouvé</b> dans les {} premiers : au mieux rang {} le {}. Raison : {}".format(TOP_LISTE, int(r["rang_jour"]), r["date_jour"].strftime("%d/%m/%Y"), r["raison_principale"]), "bad")
        bilan.append({"cas": cas["gab"], "quoi": cas.get("quoi", ""), "résultat": "NON retrouvé", "rang du jour": int(r["rang_jour"]), "date": r["date_jour"].strftime("%d/%m/%Y")})
    gc = t["num_automate"].iloc[0]
    s_ = df[(df["num_automate"] == gc) & (df["date_jour"] >= d0 - pd.Timedelta(days=60)) & (df["date_jour"] <= min(FIN, d0 + pd.Timedelta(days=10)))]
    _al = dfd[(dfd["num_automate"] == gc) & (dfd["rang_jour"] <= TOP_LISTE) & dfd["date_jour"].isin(s_["date_jour"])]
    g_courbes(s_["date_jour"], [("retraits", s_["nb_retraits"], BLEU), ("habitude", s_["hab_retraits"], GRIS, "dash")], "GAB {} : retraits par jour, habitude, et jours dans la liste du jour (orange)".format(gc),
              y_titre="retraits", hauteur=300, lignes_v=[(d0, "incident")],
              marqueurs=[("dans la liste du jour", _al["date_jour"], _al["nb_retraits"], "#e36209", ["rang {} : {}".format(int(a_), b_) for a_, b_ in zip(_al["rang_jour"], _al["raison_principale"])])] if len(_al) else None)
if bilan:
    bilan = pd.DataFrame(bilan).fillna("")
    display(tableau(bilan, "Bilan des cas connus"))
    retenir(["<b>{}</b> cas retrouvé(s) sur {}.".format(int((bilan["résultat"] == "retrouvé").sum()), len(bilan))])

### Ce que le modèle voit que des règles simples ne voient pas
Les journées très inhabituelles sur un seul indicateur entrent-elles bien dans la liste du jour ? Et combien de journées le modèle
signale-t-il sans aucun écart extrême ? Ce sont les combinaisons de petits écarts, son apport propre.

In [ ]:
ext = dfd["flag_statistique"] == 1
tab_val = pd.DataFrame({"situation": ["au moins un écart extrême (|écart| ≥ 4)", "atypiques selon le modèle", "les deux", "atypiques sans écart extrême (combinaisons subtiles)",
                                      "écart extrême non marqué atypique", "écart extrême dans la liste du jour (top {})".format(TOP_LISTE)],
                        "GAB-jours": [int(ext.sum()), int(aty.sum()), int((ext & aty).sum()), int((~ext & aty).sum()), int((ext & ~aty).sum()), int((ext & (dfd["rang_jour"] <= TOP_LISTE)).sum())],
                        "GAB distincts": [dfd.loc[ext, "num_automate"].nunique(), dfd.loc[aty, "num_automate"].nunique(), dfd.loc[ext & aty, "num_automate"].nunique(), dfd.loc[~ext & aty, "num_automate"].nunique(),
                                          dfd.loc[ext & ~aty, "num_automate"].nunique(), dfd.loc[ext & (dfd["rang_jour"] <= TOP_LISTE), "num_automate"].nunique()]})
display(tableau(tab_val, "Écarts extrêmes et modèle"))
retenir(["<b>{}</b> des GAB-jours à écart extrême entrent dans la liste du jour : le classement final (score du modèle ou force du signal) ne laisse pas passer les pics isolés.".format(
             pct((ext & (dfd["rang_jour"] <= TOP_LISTE)).sum() / max(1, ext.sum()), 0)),
         "<b>{}</b> GAB-jours sont atypiques sans aucun écart extrême : des combinaisons de petits écarts que des règles une à une ne verraient pas.".format(nb(int((~ext & aty).sum())))])

# Partie D — Les livrables
## 11. Le bilan par année et le fichier Excel
**Comment lire** : un **GAB-jour** est un GAB un jour donné ; « GAB différents » compte les automates. Un GAB atypique 10 jours compte 10
GAB-jours et 1 GAB. Le fichier Excel reprend toute la période ; la colonne « avis métier » de l'onglet Top du jour est faite pour recueillir
le retour du terrain.

In [ ]:
dfd["annee"] = dfd["date_jour"].dt.year
ga = dfd.groupby("annee")
syn = pd.DataFrame({"jours": ga["date_jour"].nunique(), "GAB": ga["num_automate"].nunique(), "GAB-jours": ga.size(), "GAB-jours atypiques": ga["est_anomalie"].sum(),
                    "atypiques par jour": (ga["est_anomalie"].sum() / ga["date_jour"].nunique()).round(1), "part des GAB-jours atypiques (%)": (100 * ga["est_anomalie"].mean()).round(2),
                    "GAB distincts atypiques au moins 1 jour": dfd[aty].groupby("annee")["num_automate"].nunique(),
                    "GAB distincts atypiques 3 jours ou plus": dfd.groupby(["annee", "num_automate"])["est_anomalie"].sum().ge(3).groupby("annee").sum()}).fillna(0)
syn["GAB jamais atypiques"] = syn["GAB"] - syn["GAB distincts atypiques au moins 1 jour"]
syn["raison dominante"] = dfd[aty].groupby("annee")["raison_principale"].agg(lambda s: s.value_counts().index[0] if len(s) else "")
for c in ["GAB-jours atypiques", "GAB distincts atypiques au moins 1 jour", "GAB distincts atypiques 3 jours ou plus", "GAB jamais atypiques"]:
    syn[c] = syn[c].astype(int)
bandeau("Synthèse par année", "GAB-jours atypiques et GAB distincts concernés")
_sy = syn.drop(columns=["raison dominante"]).T.reset_index().rename(columns={"index": "indicateur"})
_sy.columns = [str(c) for c in _sy.columns]
display(tableau(_sy, "Une colonne par année"))
display(tableau(syn[["raison dominante"]].reset_index().rename(columns={"annee": "année"}), "La raison dominante de chaque année"))
pj = dfd.groupby("date_jour").agg(atypiques=("est_anomalie", "sum"), gab=("num_automate", "size"))
t1 = dfd[dfd["rang_jour"] == 1].set_index("date_jour")
pj["GAB le plus atypique"] = t1["num_automate"]
pj["sa raison"] = t1["raison_principale"]
g_barres(pj.index.strftime("%d/%m/%Y"), [("GAB-jours atypiques", pj["atypiques"].values, BLEU)], "GAB-jours atypiques par jour sur la période (écart plus rare que 1 sur {})".format(nb(10 ** SEUIL_RARETE)), hauteur=300)
_mm = dfd.assign(mois=dfd["date_jour"].dt.to_period("M").astype(str))
_m2 = pd.DataFrame({"GAB-jours atypiques": _mm[_mm["est_anomalie"] == 1].groupby("mois").size(), "GAB distincts": _mm[_mm["est_anomalie"] == 1].groupby("mois")["num_automate"].nunique()}).fillna(0)
g_barres(_m2.index, [("GAB-jours atypiques", _m2["GAB-jours atypiques"].values, BLEU_CLAIR), ("GAB distincts", _m2["GAB distincts"].values, BLEU)], "Par mois : GAB-jours atypiques et GAB distincts concernés", hauteur=300)
retenir(["Sur <b>{}</b> jours, <b>{}</b> GAB-jours atypiques ({} par jour), concernant <b>{}</b> GAB distincts ; <b>{}</b> GAB n'ont jamais été atypiques.".format(
             nb(NJ), nb(aty.sum()), nb(aty.sum() / max(1, NJ), 1), nb(dfd.loc[aty, "num_automate"].nunique()), nb(dfd["num_automate"].nunique() - dfd.loc[aty, "num_automate"].nunique())),
         "La raison dominante est <b>{}</b>.".format(dfd.loc[aty, "raison_principale"].value_counts().index[0]) if aty.any() else ""])

In [ ]:
LIMITE_EXCEL = 1048575


def classeur(feuilles, nom):
    for moteur in ("openpyxl", "xlsxwriter"):
        flux = BytesIO()
        try:
            with pd.ExcelWriter(flux, engine=moteur) as w:
                for onglet, d in feuilles:
                    d.head(LIMITE_EXCEL).to_excel(w, sheet_name=onglet[:31], index=False)
            return flux.getvalue(), nom, moteur
        except ImportError:
            continue
    flux = BytesIO()
    with zipfile.ZipFile(flux, "w", zipfile.ZIP_DEFLATED) as zf:
        for onglet, d in feuilles:
            zf.writestr(unicodedata.normalize("NFKD", onglet).encode("ascii", "ignore").decode() + ".csv", "﻿" + d.to_csv(index=False, sep=";"))
    return flux.getvalue(), nom.replace(".xlsx", "") + "_csv.zip", "zip de CSV"


def lien_telechargement(octets, filename):
    mime = "application/zip" if filename.endswith(".zip") else "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    return HTML(CSS + '<div class="g-note ok"><a href="data:{};base64,{}" download="{}" style="font-size:1.1em"><b>Télécharger {}</b></a> ({} Mo)</div>'.format(
        mime, base64.b64encode(octets).decode(), filename, filename, nb(len(octets) / 1e6, 1)))


def jj(d):
    return d.dt.strftime("%Y-%m-%d")


COL_X = {"date_jour": "jour", "rang_jour": "rang du jour", "num_automate": "GAB", "code_site_implantation": "code site", "ville": "ville", "code_postal": "code postal", "latitude": "latitude", "longitude": "longitude",
         "famille": "famille", "rarete": "rareté de l'écart", "score_final": "score", "score_risque": "score de risque", "score_signal": "force du signal", "est_anomalie": "atypique", "raison_principale": "raison principale",
         "detail_atypisme": "détail", "nb_retraits": "retraits", "hab_retraits": "habitude (retraits)", "montant_total": "montant retiré (€)", "nb_inaboutis": "inaboutis", "nb_captures": "captures",
         "nb_retraits_repetes_10min": "retraits refaits < 10 min", "nb_exotique": "cartes étrangères", "nb_nuit": "retraits de nuit", "score_risque_global": "score de risque (modèle global)", "flag_statistique": "écart extrême"}
cols_x = ["date_jour", "rang_jour", "num_automate"] + [c for c in ["code_site_implantation", "ville", "code_postal", "latitude", "longitude"] if c in dfd.columns] + \
         ["famille", "rarete", "score_final", "score_risque", "score_signal", "est_anomalie", "raison_principale", "detail_atypisme", "nb_retraits", "hab_retraits", "montant_total", "nb_inaboutis", "nb_captures", "nb_retraits_repetes_10min", "nb_exotique", "nb_nuit"]
top_x = completer_raisons(dfd[(dfd["rang_jour"] <= TOP_LISTE) & ((dfd["est_anomalie"] == 1) | dfd["arret_signale"])].sort_values(["date_jour", "rang_jour"], ascending=[False, True]).copy())[cols_x].copy()
top_x["date_jour"] = jj(top_x["date_jour"])
top_x["avis_metier"] = ""
aty_x = dfd[aty].sort_values(["date_jour", "score_risque"], ascending=[False, False])[cols_x].copy()
aty_x["date_jour"] = jj(aty_x["date_jour"])
pj_x = pj.reset_index()
pj_x["date_jour"] = jj(pj_x["date_jour"])
kpi_x = kpi_parc.reset_index().sort_values("date_jour", ascending=False)
kpi_x["date_jour"] = jj(kpi_x["date_jour"])
pod_x = podiums.sort_values(["date_jour", "indicateur", "rang"], ascending=[False, True, True]).copy()
pod_x["date_jour"] = jj(pod_x["date_jour"])
lisez_moi = pd.DataFrame({"rubrique": ["Période", "Dernier jour complet", "Jours écartés en fin de fiche", "GAB", "Familles", "GAB-jour", "Score de risque", "Force du signal", "Score", "Rang du jour", "Atypique", "Généré le"],
                          "contenu": ["du {} au {} ({} jours)".format(DEBUT_FENETRE.strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y"), NJ), FIN.strftime("%d/%m/%Y"),
                                      ", ".join(d_.strftime("%d/%m/%Y") for d_ in sorted(ecartes)) or "aucun", "{} GAB".format(dfd["num_automate"].nunique()), "{} familles K-Means sur le profil de chaque GAB".format(dfd["cluster"].nunique()),
                                      "un GAB un jour donné ; un GAB atypique 10 jours = 10 GAB-jours et 1 GAB distinct",
                                      "rang du GAB-jour dans sa famille selon l'Isolation Forest (0 à 1 ; 1 = le plus inhabituel)", "rareté de l'écart le plus rare du jour : 3 = 1 cas sur 1 000, 4 = 1 sur 10 000",
                                      "le plus rare de la force du signal et du modèle (en appoint) : c'est lui qui classe", "1 = le GAB le plus atypique du jour", "1 si le score atteint {} (écart plus rare que 1 sur {})".format(SEUIL_RARETE, nb(10 ** SEUIL_RARETE)),
                                      pd.Timestamp.now().strftime("%d/%m/%Y %H:%M")]})
feuilles = [("Lisez-moi", lisez_moi), ("Par année", syn.reset_index().rename(columns={"annee": "année"})), ("Chiffres du jour", kpi_x.round(1)), ("Podiums par jour", pod_x),
            ("Par jour", pj_x.rename(columns={"date_jour": "jour", "atypiques": "GAB-jours atypiques", "gab": "GAB"})),
            ("Top du jour", top_x.round(3).rename(columns=dict(COL_X, avis_metier="avis métier"))), ("GAB", df_gab), ("Atypiques", aty_x.round(3).rename(columns=COL_X)),
            ("Familles", tab_fam.reset_index().rename(columns=dict(LIB_PROFIL, cluster="famille")))]
display(tableau(pd.DataFrame([(o_, len(d_), d_.shape[1]) for o_, d_ in feuilles], columns=["onglet", "lignes", "colonnes"]), "Les onglets du classeur"))
octets, nom_fichier, moteur = classeur(feuilles, NOM_EXCEL_SORTIE)
print("{} : {} Mo ({}).".format(nom_fichier, nb(len(octets) / 1e6, 1), moteur))
if len(octets) <= 50e6:
    display(lien_telechargement(octets, nom_fichier))
if NOM_DOSSIER_DATAIKU:
    dataiku.Folder(NOM_DOSSIER_DATAIKU).upload_stream(nom_fichier, BytesIO(octets))
    note("Déposé dans le dossier Dataiku <b>{}</b> : {}".format(NOM_DOSSIER_DATAIKU, nom_fichier), "ok")
if DATASET_SORTIE:
    s_ = dfd[cols_x + ["score_risque_global", "flag_statistique"]].copy()
    s_["date_jour"] = jj(s_["date_jour"])
    dataiku.Dataset(DATASET_SORTIE).write_with_schema(s_.rename(columns=COL_X))
    note("GAB-jours scorés écrits dans {} ({} lignes).".format(DATASET_SORTIE, nb(len(s_))), "ok")